# Day 16 — Initial Out-of-Time Evaluation

Evaluate the frozen historical baseline on the six strictly future windows.

In [ ]:
# Imports
import json
from pathlib import Path

In [ ]:
# Project paths
def find_project_root():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "README.md").exists():
            return path
    raise RuntimeError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

def resolve_artifact_paths(value):
    """Rebase legacy project paths in loaded metadata; never rewrite artifacts.

    Keep all non-path values, model identifiers, and scientific settings intact.
    Support the original project name and the public repository name.
    """
    if isinstance(value, dict):
        return {key: resolve_artifact_paths(item) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve_artifact_paths(item) for item in value]
    if isinstance(value, str):
        normalized = value.replace("\\", "/")
        for name in ("FinGPT_Project", "FinGPT-Temporal-Generalization"):
            marker = "/" + name + "/"
            if marker in normalized and "\n" not in normalized:
                # Only standalone filesystem references, never prose containing a path.
                prefix, relative = normalized.split(marker, 1)
                if prefix.startswith("/") or (len(prefix) > 1 and prefix[1] == ":"):
                    if not any(char.isspace() for char in relative):
                        return str(PROJECT_ROOT / relative)
    return value

def load_artifact_json(stream, *args, **kwargs):
    return resolve_artifact_paths(json.load(stream, *args, **kwargs))


In [ ]:
# Reproducibility settings
# Original seeds and deterministic settings remain at their original workflow
# positions below, including stage-specific RNG resets. No new seed is introduced.

## FinGPT: Financial Language Models under Temporal Distribution Shift

### Scientific Objective
Evaluate the **frozen historical LoRA model** on strictly future Dataset-B observations using the **Day14 frozen chronological protocol**, without retraining, calibration, threshold tuning, window redefinition, or any interpretation-driven modification.

### Research Question

> **How does predictive performance change across strictly future temporal windows?**

### Day 16 Scientific Boundary

- **Day 16 = Measurement**
- **Day 17 = Diagnosis**
- **Day 18 = Controlled robustness intervention**

Day 16 is a descriptive chronological evaluation.  
It does **not** claim that time itself causally produces performance degradation.

### Frozen Evaluation Structure

- Historical cutoff: **2026-01-01 00:00:00 UTC**
- Historical Validation:
  - pooled **Nov–Dec 2025**
  - used only as a **baseline anchor**
- Future evaluation population:
  - **20,002 observations**
- Future windows:
  - **F1–F5:** full monthly windows, Jan–May 2026
  - **F6:** partial June 2026 sensitivity window
- Primary endpoint:
  - **Macro-F1**
- Secondary endpoints:
  - Accuracy
  - Weighted-F1
  - Class-specific recall
- Diagnostics:
  - Confusion transitions
  - Prediction distribution
  - Source composition

### Label Provenance Caveat

All Day16 performance metrics are evaluated against the **frozen Dataset-B sentiment benchmark labels**, which are **LLM-generated benchmark labels rather than conventional human gold labels**.

### Core Rule

> **One frozen model. One fixed inference pipeline. One immutable future population. No adaptation after observing results.**

## Stage 0 — Environment and Deterministic Setup

Initialize the Day16 evaluation environment before loading any frozen protocol artifacts or model checkpoints.

This stage establishes:

- reproducible random seeds;
- the available computation device;
- package/runtime versions;
- common paths used by the Day16 notebook.

No model loading, inference, or evaluation is performed at this stage.

In [ ]:
# Environment and deterministic setup
import os
import sys
import json
import random
import platform
from pathlib import Path

import numpy as np
import pandas as pd

import torch

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Deterministic behavior where supported
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Core project paths
# PROJECT_ROOT is discovered in the project-path setup cell.
RESULTS_ROOT = PROJECT_ROOT / "results"
NOTEBOOK_ROOT = PROJECT_ROOT / "notebooks"

DAY14_DIR = RESULTS_ROOT / "day14"
DAY15_DIR = RESULTS_ROOT / "day15"
DAY16_DIR = RESULTS_ROOT / "day16"

DAY16_DIR.mkdir(parents=True, exist_ok=True)

# Environment report

print("=" * 88)
print("DAY 16 — ENVIRONMENT AND DETERMINISTIC SETUP")
print("=" * 88)

print(f"Python version      : {sys.version.split()[0]}")
print(f"Platform            : {platform.platform()}")
print(f"PyTorch version     : {torch.__version__}")
print(f"NumPy version       : {np.__version__}")
print(f"Pandas version      : {pd.__version__}")

print("-" * 88)

print(f"CUDA available      : {torch.cuda.is_available()}")
print(f"Selected device     : {DEVICE}")

if torch.cuda.is_available():
    print(f"CUDA device count   : {torch.cuda.device_count()}")
    print(f"CUDA device name    : {torch.cuda.get_device_name(0)}")
    print(f"CUDA runtime        : {torch.version.cuda}")

print("-" * 88)

print(f"Random seed         : {SEED}")
print(f"Project root        : {PROJECT_ROOT}")
print(f"Day14 results dir   : {DAY14_DIR}")
print(f"Day15 results dir   : {DAY15_DIR}")
print(f"Day16 results dir   : {DAY16_DIR}")
print(f"Day16 dir exists    : {DAY16_DIR.exists()}")

print("=" * 88)

### Stage A1 — Load Frozen Protocol and Model Metadata

Load the Day14 chronological evaluation protocol and Day15 frozen-model records that define the Day16 experiment.

The following artifacts are treated as the source of truth:

**Day14**
- `benchmark_population_manifest.json`
- `chronological_split_spec.json`
- `future_window_manifest.csv`
- `source_composition_by_split.csv`
- `evaluation_protocol.json`
- `lora_training_protocol.json`
- `final_boundary_leakage_audit.json`

**Day15**
- `day15_summary.json`
- `frozen_model_manifest.json`
- `historical_train_manifest.json`
- `historical_validation_manifest.json`
- `historical_validation_metrics.json`
- `training_data_audit.json`
- `checkpoint_selection.csv`

This step only loads and inspects frozen metadata.  
No values are modified, inferred, or redefined.

In [ ]:
# Load frozen Day14 protocol and Day15 model metadata

# File paths

DAY14_PATHS = {
    "benchmark_population_manifest":
        DAY14_DIR / "benchmark_population_manifest.json",

    "chronological_split_spec":
        DAY14_DIR / "chronological_split_spec.json",

    "future_window_manifest":
        DAY14_DIR / "future_window_manifest.csv",

    "source_composition_by_split":
        DAY14_DIR / "source_composition_by_split.csv",

    "evaluation_protocol":
        DAY14_DIR / "evaluation_protocol.json",

    "lora_training_protocol":
        DAY14_DIR / "lora_training_protocol.json",

    "final_boundary_leakage_audit":
        DAY14_DIR / "final_boundary_leakage_audit.json",
}

DAY15_PATHS = {
    "day15_summary":
        DAY15_DIR / "day15_summary.json",

    "frozen_model_manifest":
        DAY15_DIR / "frozen_model_manifest.json",

    "historical_train_manifest":
        DAY15_DIR / "historical_train_manifest.json",

    "historical_validation_manifest":
        DAY15_DIR / "historical_validation_manifest.json",

    "historical_validation_metrics":
        DAY15_DIR / "historical_validation_metrics.json",

    "training_data_audit":
        DAY15_DIR / "training_data_audit.json",

    "checkpoint_selection":
        DAY15_DIR / "checkpoint_selection.csv",
}
# Verify required files exist

all_required_paths = {**DAY14_PATHS, **DAY15_PATHS}

missing_files = [
    str(path)
    for path in all_required_paths.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required frozen artifacts:\n" +
        "\n".join(missing_files)
    )

# Helpers
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return load_artifact_json(f)

# Load Day14
benchmark_population_manifest = load_json(
    DAY14_PATHS["benchmark_population_manifest"]
)

chronological_split_spec = load_json(
    DAY14_PATHS["chronological_split_spec"]
)

future_window_manifest = pd.read_csv(
    DAY14_PATHS["future_window_manifest"]
)

source_composition_by_split = pd.read_csv(
    DAY14_PATHS["source_composition_by_split"]
)

evaluation_protocol = load_json(
    DAY14_PATHS["evaluation_protocol"]
)

lora_training_protocol = load_json(
    DAY14_PATHS["lora_training_protocol"]
)

final_boundary_leakage_audit = load_json(
    DAY14_PATHS["final_boundary_leakage_audit"]
)

# Load Day15
day15_summary = load_json(
    DAY15_PATHS["day15_summary"]
)

frozen_model_manifest = load_json(
    DAY15_PATHS["frozen_model_manifest"]
)

historical_train_manifest = load_json(
    DAY15_PATHS["historical_train_manifest"]
)

historical_validation_manifest = load_json(
    DAY15_PATHS["historical_validation_manifest"]
)

historical_validation_metrics = load_json(
    DAY15_PATHS["historical_validation_metrics"]
)

training_data_audit = load_json(
    DAY15_PATHS["training_data_audit"]
)

checkpoint_selection = pd.read_csv(
    DAY15_PATHS["checkpoint_selection"]
)

# Compact structure report
print("=" * 100)
print("DAY 16 — FROZEN PROTOCOL / MODEL METADATA LOADED")
print("=" * 100)

print("\n[DAY 14 JSON STRUCTURES]")

for name, obj in [
    ("benchmark_population_manifest", benchmark_population_manifest),
    ("chronological_split_spec", chronological_split_spec),
    ("evaluation_protocol", evaluation_protocol),
    ("lora_training_protocol", lora_training_protocol),
    ("final_boundary_leakage_audit", final_boundary_leakage_audit),
]:
    print(f"\n{name}")
    print(" keys:", list(obj.keys()))

print("\n" + "-" * 100)

print("\n[DAY 14 TABLES]")

print("\nfuture_window_manifest")
print(" shape   :", future_window_manifest.shape)
print(" columns :", list(future_window_manifest.columns))
display(future_window_manifest)

print("\nsource_composition_by_split")
print(" shape   :", source_composition_by_split.shape)
print(" columns :", list(source_composition_by_split.columns))
display(source_composition_by_split)

print("\n" + "-" * 100)

print("\n[DAY 15 JSON STRUCTURES]")

for name, obj in [
    ("day15_summary", day15_summary),
    ("frozen_model_manifest", frozen_model_manifest),
    ("historical_train_manifest", historical_train_manifest),
    ("historical_validation_manifest", historical_validation_manifest),
    ("historical_validation_metrics", historical_validation_metrics),
    ("training_data_audit", training_data_audit),
]:
    print(f"\n{name}")
    print(" keys:", list(obj.keys()))

print("\ncheckpoint_selection")
print(" shape   :", checkpoint_selection.shape)
print(" columns :", list(checkpoint_selection.columns))
display(checkpoint_selection)

print("\n" + "=" * 100)
print("All required Day14 / Day15 frozen artifacts loaded successfully.")
print("=" * 100)

### Stage A3 — Frozen Protocol Integrity Gate

Verify that Day16 is starting from the exact frozen Day14/Day15 experimental state.

This gate checks:

- chronological cutoff and split ordering;
- frozen future population and F1–F6 definitions;
- endpoint hierarchy and label order;
- frozen model identity and checkpoint selection;
- inference settings;
- historical-validation baseline;
- future-data firewall;
- Day15 training-data audit;
- current accessibility of the frozen adapter after normalizing the historical Google Drive mount path.

If any protocol-critical check fails, Day16 stops before future data are materialized or evaluated.

On PASS, this cell writes the first permanent Day16 artifact:

`future_inference_manifest.json`

In [ ]:
# Stage A frozen protocol integrity gate
from datetime import datetime, timezone

print("=" * 104)
print("DAY 16 — STAGE A: FROZEN PROTOCOL INTEGRITY GATE")
print("=" * 104)

# Constants inherited from frozen protocol
EXPECTED_CUTOFF = "2026-01-01T00:00:00+00:00"
EXPECTED_FUTURE_N = 20002
EXPECTED_TRAIN_N = 13539
EXPECTED_VALID_N = 5859
EXPECTED_WINDOWS = ["F1", "F2", "F3", "F4", "F5", "F6"]
EXPECTED_LABEL_ORDER = ["negative", "neutral", "positive"]
EXPECTED_PRIMARY_ENDPOINT = "macro_f1"
EXPECTED_SELECTED_STEP = 200

LABEL_PROVENANCE_CAVEAT = (
    "Day16 performance is evaluated against the frozen Dataset-B sentiment "
    "benchmark labels. These labels are LLM-generated benchmark labels rather "
    "than conventional human gold labels."
)

# Resolve frozen artifact locations within this repository.
def resolve_recorded_path(value):
    return None if value is None else Path(resolve_artifact_paths(value))


frozen_adapter_recorded = frozen_model_manifest["model_identity"][
    "frozen_model_directory"
]

selected_checkpoint_recorded = frozen_model_manifest[
    "checkpoint_selection"
]["selected_checkpoint_path"]

FROZEN_ADAPTER_PATH = resolve_recorded_path(
    frozen_adapter_recorded
)

SELECTED_CHECKPOINT_PATH = resolve_recorded_path(
    selected_checkpoint_recorded
)
# Individual integrity checks

checks = {}

# Chronological protocol
checks["cutoff_matches_day14"] = (
    chronological_split_spec["historical_cutoff"]["cutoff_timestamp_utc"]
    == EXPECTED_CUTOFF
)

checks["historical_train_n_matches"] = (
    chronological_split_spec["historical_train"]["n_observations"]
    == EXPECTED_TRAIN_N
)

checks["historical_validation_n_matches"] = (
    chronological_split_spec["historical_validation"]["n_observations"]
    == EXPECTED_VALID_N
)

checks["future_n_matches"] = (
    chronological_split_spec["strictly_future_total"]["n_observations"]
    == EXPECTED_FUTURE_N
)

checks["chronological_protocol_locked"] = (
    chronological_split_spec["freeze_status"] == "LOCKED"
)

# Future windows
actual_windows = [
    x["future_window"]
    for x in chronological_split_spec["future_windows"]
]

checks["future_windows_match"] = (
    actual_windows == EXPECTED_WINDOWS
)

checks["f6_marked_partial"] = (
    chronological_split_spec["future_windows"][-1]["future_window"] == "F6"
    and chronological_split_spec["future_windows"][-1]["is_partial_month"] is True
)

# Evaluation protocol
checks["evaluation_protocol_locked"] = (
    evaluation_protocol["freeze_status"]
    == "LOCKED_BEFORE_MODEL_EVALUATION"
)

checks["primary_endpoint_is_macro_f1"] = (
    evaluation_protocol["metrics"]["primary"]["name"]
    == EXPECTED_PRIMARY_ENDPOINT
)

checks["label_order_matches"] = (
    evaluation_protocol["label_schema"]["label_order"]
    == EXPECTED_LABEL_ORDER
)

checks["same_metric_implementation"] = (
    evaluation_protocol["evaluation_scope"][
        "same_metric_implementation_for_all_splits"
    ] is True
)

# Frozen model
checks["model_id_is_M_H"] = (
    frozen_model_manifest["model_identity"]["model_id"]
    == "M_H"
)

checks["model_freeze_status_valid"] = (
    frozen_model_manifest["model_identity"]["freeze_status"]
    == "FROZEN_BEFORE_FUTURE_EVALUATION"
)

checks["selected_checkpoint_step_matches"] = (
    frozen_model_manifest["checkpoint_selection"][
        "selected_checkpoint_step"
    ] == EXPECTED_SELECTED_STEP
)

checks["checkpoint_selected_without_future_metrics"] = (
    frozen_model_manifest["checkpoint_selection"][
        "future_metrics_used"
    ] is False
)

# Inference protocol
day14_inference = lora_training_protocol["inference"]
day15_inference = frozen_model_manifest["inference_protocol"]

checks["inference_eval_mode"] = (
    day14_inference["model_mode"] == "eval"
    and day15_inference["model_mode"] == "eval"
)

checks["inference_left_padding"] = (
    day14_inference["tokenizer_padding_side"] == "left"
    and day15_inference["padding_side"] == "left"
)

checks["inference_deterministic_generation"] = (
    day14_inference["generation"]["do_sample"] is False
    and day15_inference["do_sample"] is False
)

checks["inference_max_new_tokens_match"] = (
    day14_inference["generation"]["max_new_tokens"]
    == day15_inference["max_new_tokens"]
    == 4
)

# Historical validation anchor
checks["validation_baseline_n_matches"] = (
    historical_validation_metrics["n_observations"]
    == EXPECTED_VALID_N
)

checks["validation_checkpoint_step_matches"] = (
    historical_validation_metrics["selected_checkpoint_step"]
    == EXPECTED_SELECTED_STEP
)

checks["validation_used_no_future_observations"] = (
    historical_validation_metrics["future_observations_used"]
    == 0
)

# Future firewall
future_firewall = frozen_model_manifest["future_firewall"]

checks["future_training_use_zero"] = (
    future_firewall["future_observations_used_for_training"] == 0
)

checks["future_validation_use_zero"] = (
    future_firewall["future_observations_used_for_validation"] == 0
)

checks["future_checkpoint_selection_use_false"] = (
    future_firewall["future_metrics_used_for_checkpoint_selection"] is False
)

checks["day15_future_evaluation_not_started"] = (
    future_firewall["future_evaluation_status"]
    == "NOT_STARTED_DAY15"
)

# Day15 training audit
checks["day15_training_audit_passed"] = (
    training_data_audit["all_checks_pass"] is True
    and training_data_audit["protocol_status"] == "PASS"
)

# Physical frozen model artifacts
checks["frozen_adapter_directory_exists"] = (
    FROZEN_ADAPTER_PATH.exists()
)

checks["selected_checkpoint_directory_exists"] = (
    SELECTED_CHECKPOINT_PATH.exists()
)

# Disclosure requirement
checks["label_provenance_disclosed"] = bool(
    LABEL_PROVENANCE_CAVEAT.strip()
)
# Overall PASS / FAIL
all_checks_pass = all(checks.values())

for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<5} — {name}")

print("-" * 104)

print(f"Recorded frozen adapter path : {frozen_adapter_recorded}")
print(f"Runtime frozen adapter path  : {FROZEN_ADAPTER_PATH}")
print(f"Adapter directory exists     : {FROZEN_ADAPTER_PATH.exists()}")

print()

print(f"Recorded checkpoint path     : {selected_checkpoint_recorded}")
print(f"Runtime checkpoint path      : {SELECTED_CHECKPOINT_PATH}")
print(f"Checkpoint directory exists  : {SELECTED_CHECKPOINT_PATH.exists()}")

print("-" * 104)

# Write permanent Day16 inference manifest only on PASS
future_inference_manifest = {
    "day": 16,
    "artifact": "future_inference_manifest",
    "project": "Financial Language Models under Temporal Distribution Shift",

    "model": {
        "model_id": frozen_model_manifest["model_identity"]["model_id"],
        "base_model": frozen_model_manifest["model_identity"]["base_model"],
        "frozen_adapter_recorded_path": frozen_adapter_recorded,
        "frozen_adapter_runtime_path": str(FROZEN_ADAPTER_PATH),
        "selected_checkpoint_step": frozen_model_manifest[
            "checkpoint_selection"
        ]["selected_checkpoint_step"],
        "selected_checkpoint_recorded_path": selected_checkpoint_recorded,
        "selected_checkpoint_runtime_path": str(
            SELECTED_CHECKPOINT_PATH
        ),
        "freeze_status": frozen_model_manifest[
            "model_identity"
        ]["freeze_status"],
    },

    "chronological_protocol": {
        "historical_cutoff_utc": chronological_split_spec[
            "historical_cutoff"
        ]["cutoff_timestamp_utc"],
        "historical_train_n": EXPECTED_TRAIN_N,
        "historical_validation_n": EXPECTED_VALID_N,
        "strictly_future_n": EXPECTED_FUTURE_N,
        "future_windows": EXPECTED_WINDOWS,
        "ordering_rule": chronological_split_spec["ordering_rule"],
        "F6_is_partial_month": True,
    },

    "evaluation_protocol": {
        "primary_endpoint": evaluation_protocol[
            "metrics"
        ]["primary"]["name"],
        "secondary_endpoints": [
            x["name"]
            for x in evaluation_protocol["metrics"]["secondary"]
        ],
        "class_specific_metrics": evaluation_protocol[
            "metrics"
        ]["class_specific"],
        "label_order": evaluation_protocol[
            "label_schema"
        ]["label_order"],
        "priority_failure_mode": evaluation_protocol[
            "confusion_analysis"
        ]["priority_failure_mode"],
    },

    "inference_protocol": {
        "model_mode": day15_inference["model_mode"],
        "padding_side": day15_inference["padding_side"],
        "do_sample": day15_inference["do_sample"],
        "max_new_tokens": day15_inference["max_new_tokens"],
        "valid_predictions": day15_inference["valid_predictions"],
        "max_seq_length": frozen_model_manifest[
            "sequence_protocol"
        ]["max_seq_length"],
        "prompt_template": frozen_model_manifest[
            "sequence_protocol"
        ]["prompt_template"],
    },

    "historical_validation_anchor": {
        "role": "pooled_reference_anchor",
        "n": historical_validation_metrics["n_observations"],
        "accuracy": historical_validation_metrics["accuracy"],
        "macro_f1": historical_validation_metrics["macro_f1"],
        "weighted_f1": historical_validation_metrics["weighted_f1"],
        "negative_recall": historical_validation_metrics[
            "class_recall"
        ]["negative"],
        "neutral_recall": historical_validation_metrics[
            "class_recall"
        ]["neutral"],
        "positive_recall": historical_validation_metrics[
            "class_recall"
        ]["positive"],
    },

    "future_firewall": {
        "future_observations_used_for_training": 0,
        "future_observations_used_for_validation": 0,
        "future_metrics_used_for_checkpoint_selection": False,
        "adaptation_performed": False,
    },

    "label_provenance_caveat": LABEL_PROVENANCE_CAVEAT,

    "stage_a_integrity_checks": checks,
    "stage_a_all_checks_pass": all_checks_pass,

    "created_utc": datetime.now(timezone.utc).isoformat(),
}

manifest_path = DAY16_DIR / "future_inference_manifest.json"

if all_checks_pass:
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(
            future_inference_manifest,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("STAGE A STATUS : PASS — FROZEN PROTOCOL VERIFIED")
    print(f"Manifest saved : {manifest_path}")
else:
    print("STAGE A STATUS : FAIL — STOP BEFORE FUTURE EVALUATION")

print("=" * 104)

if not all_checks_pass:
    failed_checks = [
        name for name, passed in checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage A integrity gate failed:\n- "
        + "\n- ".join(failed_checks)
    )

### Stage B2 — Load and Verify the Frozen 39,400-Observation Benchmark Population

Reconstruct the exact Day14 benchmark population from the frozen upstream Dataset-B artifact.

The benchmark definition is inherited without modification:

- upstream source: Day07 `temporal_dataset_clean.csv`;
- benchmark rule: Day13 dense temporal regime only;
- expected population size: **39,400**;
- frozen time span:
  - start: `2025-06-03T21:06:16+00:00`
  - end: `2026-06-05T12:59:13+00:00`
- required columns:
  - `message_id`
  - `channel_username`
  - `date`
  - `text`
  - `sentiment`
- observation ID:
  - `channel_username::message_id`

This cell validates the benchmark population before any historical/future split is materialized.

No new filtering rule, cutoff, or population definition is introduced.

In [ ]:
# Load and verify frozen 39,400-row benchmark population
print("=" * 104)
print("DAY 16 — STAGE B2: LOAD FROZEN BENCHMARK POPULATION")
print("=" * 104)

# Resolve frozen upstream path
recorded_dataset_path = benchmark_population_manifest[
    "upstream_provenance"
]["dataset_source"]

DATASET_B_SOURCE = resolve_recorded_path(
    recorded_dataset_path
)

print(f"Recorded source path : {recorded_dataset_path}")
print(f"Runtime source path  : {DATASET_B_SOURCE}")
print(f"Source file exists   : {DATASET_B_SOURCE.exists()}")

if not DATASET_B_SOURCE.exists():
    raise FileNotFoundError(
        f"Frozen Dataset-B source not found: {DATASET_B_SOURCE}"
    )

# Load upstream Dataset-B artifact

dataset_b_raw = pd.read_csv(DATASET_B_SOURCE)

print("-" * 104)
print(f"Upstream rows        : {len(dataset_b_raw):,}")
print(f"Upstream columns     : {len(dataset_b_raw.columns)}")

# Verify required frozen columns
required_columns = benchmark_population_manifest["required_columns"]

missing_required_columns = [
    col for col in required_columns
    if col not in dataset_b_raw.columns
]

if missing_required_columns:
    raise RuntimeError(
        "Missing required frozen columns: "
        + ", ".join(missing_required_columns)
    )
# Normalize fields according to frozen protocol
dataset_b_raw = dataset_b_raw.copy()

dataset_b_raw["date"] = pd.to_datetime(
    dataset_b_raw["date"],
    utc=True,
    errors="coerce"
)

dataset_b_raw["sentiment"] = (
    dataset_b_raw["sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
)

dataset_b_raw["channel_username"] = (
    dataset_b_raw["channel_username"]
    .astype(str)
    .str.strip()
)

dataset_b_raw["observation_id"] = (
    dataset_b_raw["channel_username"].astype(str)
    + "::"
    + dataset_b_raw["message_id"].astype(str)
)

# Apply ONLY the frozen Day14 benchmark time span
benchmark_start = pd.Timestamp(
    benchmark_population_manifest[
        "benchmark_population"
    ]["start_timestamp_utc"]
)

benchmark_end = pd.Timestamp(
    benchmark_population_manifest[
        "benchmark_population"
    ]["end_timestamp_utc"]
)

benchmark_population = (
    dataset_b_raw.loc[
        dataset_b_raw["date"].between(
            benchmark_start,
            benchmark_end,
            inclusive="both"
        )
    ]
    .copy()
    .sort_values("date", kind="stable")
    .reset_index(drop=True)
)

# Frozen expectations
expected_n = benchmark_population_manifest[
    "benchmark_population"
]["n_observations"]

expected_labels = benchmark_population_manifest[
    "label_schema"
]["classes"]

expected_label_counts = {
    label: benchmark_population_manifest[
        "label_schema"
    ]["distribution"][label]["count"]
    for label in expected_labels
}

expected_source_counts = {
    source: info["count"]
    for source, info in benchmark_population_manifest[
        "source_schema"
    ]["distribution"].items()
}

# Observed benchmark properties
observed_label_counts = (
    benchmark_population["sentiment"]
    .value_counts()
    .reindex(expected_labels, fill_value=0)
    .astype(int)
    .to_dict()
)

observed_source_counts = (
    benchmark_population["channel_username"]
    .value_counts()
    .astype(int)
    .to_dict()
)

observed_start = benchmark_population["date"].min()
observed_end = benchmark_population["date"].max()

# Integrity checks
benchmark_checks = {}

benchmark_checks["benchmark_n_matches"] = (
    len(benchmark_population) == expected_n == 39400
)

benchmark_checks["no_missing_timestamp"] = (
    benchmark_population["date"].notna().all()
)

benchmark_checks["observation_ids_present"] = (
    benchmark_population["observation_id"]
    .notna()
    .all()
)

benchmark_checks["observation_ids_unique"] = (
    benchmark_population["observation_id"]
    .nunique()
    == len(benchmark_population)
)

benchmark_checks["labels_valid"] = (
    set(benchmark_population["sentiment"].unique())
    <= set(expected_labels)
)

benchmark_checks["label_counts_match_manifest"] = (
    observed_label_counts == expected_label_counts
)

benchmark_checks["source_counts_match_manifest"] = (
    observed_source_counts == expected_source_counts
)

benchmark_checks["start_timestamp_matches"] = (
    observed_start == benchmark_start
)

benchmark_checks["end_timestamp_matches"] = (
    observed_end == benchmark_end
)

benchmark_checks["chronologically_sorted"] = (
    benchmark_population["date"].is_monotonic_increasing
)

print("-" * 104)

for name, passed in benchmark_checks.items():
    print(f"{'PASS' if passed else 'FAIL':<5} — {name}")

print("-" * 104)

print(f"Frozen benchmark n   : {len(benchmark_population):,}")
print(f"Start UTC            : {observed_start}")
print(f"End UTC              : {observed_end}")
print(
    f"Unique observation ID: "
    f"{benchmark_population['observation_id'].nunique():,}"
)

print("\nObserved label counts:")
for label in expected_labels:
    print(
        f"  {label:<8}: "
        f"{observed_label_counts.get(label, 0):,}"
    )

print("\nObserved source counts:")
for source, count in sorted(
    observed_source_counts.items(),
    key=lambda x: (-x[1], x[0])
):
    print(f"  {source:<24}: {count:,}")

print("-" * 104)

all_benchmark_checks_pass = all(
    benchmark_checks.values()
)

if all_benchmark_checks_pass:
    print(
        "STAGE B2 STATUS : PASS — "
        "FROZEN 39,400-ROW BENCHMARK POPULATION VERIFIED"
    )
else:
    print(
        "STAGE B2 STATUS : FAIL — "
        "STOP BEFORE CHRONOLOGICAL MATERIALIZATION"
    )

print("=" * 104)

if not all_benchmark_checks_pass:
    failed = [
        name
        for name, passed in benchmark_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage B2 benchmark verification failed:\n- "
        + "\n- ".join(failed)
    )

### Stage B3 — Materialize and Audit the Exact F1–F6 Future Population

Construct the strictly future Day16 evaluation population using only the frozen Day14 chronological protocol.

This stage must verify that:

- the total future population contains exactly **20,002 observations**;
- every future observation occurs strictly after the frozen historical cutoff;
- every observation belongs to exactly one of `F1`–`F6`;
- per-window sample counts match the Day14 frozen future-window manifest;
- per-window label counts match the frozen protocol;
- per-window time boundaries match the frozen protocol;
- observation IDs remain unique;
- source counts match the Day14 frozen source-composition manifest.

No new temporal boundaries are estimated or adjusted.

On PASS, this stage writes:

`future_population_audit.json`

The materialized `future_population` object becomes the immutable in-memory Day16 evaluation population used by the fixed inference pipeline.

In [ ]:
# Materialize exact F1–F6 future population and audit
from collections import OrderedDict

print("=" * 108)
print("DAY 16 — STAGE B3: MATERIALIZE EXACT F1–F6 FUTURE POPULATION")
print("=" * 108)

# Frozen chronological definitions

cutoff_utc = pd.Timestamp(
    chronological_split_spec[
        "historical_cutoff"
    ]["cutoff_timestamp_utc"]
)

future_specs = chronological_split_spec["future_windows"]

# Map future-window specifications by window name
future_spec_map = {
    spec["future_window"]: spec
    for spec in future_specs
}

# Start from the already-verified 39,400-row benchmark
future_population = benchmark_population.loc[
    benchmark_population["date"] >= cutoff_utc
].copy()

future_population = (
    future_population
    .sort_values("date", kind="stable")
    .reset_index(drop=True)
)

# Assign frozen future-window labels
future_population["period"] = pd.NA
future_population["period_type"] = pd.NA

for spec in future_specs:
    window = spec["future_window"]

    start = pd.Timestamp(spec["start_timestamp_utc"])
    end = pd.Timestamp(spec["end_timestamp_utc"])

    mask = future_population["date"].between(
        start,
        end,
        inclusive="both"
    )

    future_population.loc[mask, "period"] = window

    future_population.loc[
        mask,
        "period_type"
    ] = (
        "future_partial"
        if spec["is_partial_month"]
        else "future_month"
    )
# Frozen expectations
expected_future_n = chronological_split_spec[
    "strictly_future_total"
]["n_observations"]

expected_window_counts = {
    spec["future_window"]: spec["n_observations"]
    for spec in future_specs
}

expected_window_label_counts = {
    spec["future_window"]: {
        "negative": spec["negative_n"],
        "neutral": spec["neutral_n"],
        "positive": spec["positive_n"],
    }
    for spec in future_specs
}

# Day14 source-composition manifest restricted to F1–F6
frozen_future_source_manifest = (
    source_composition_by_split.loc[
        source_composition_by_split["split"].isin(
            EXPECTED_WINDOWS
        )
    ]
    .copy()
)

# Basic future-population checks
future_checks = OrderedDict()

future_checks["future_n_matches"] = (
    len(future_population)
    == expected_future_n
    == 20002
)

future_checks["all_rows_on_or_after_cutoff"] = (
    future_population["date"].ge(cutoff_utc).all()
)

future_checks["no_missing_period_assignment"] = (
    future_population["period"].notna().all()
)

future_checks["periods_are_exactly_F1_to_F6"] = (
    set(future_population["period"].dropna().unique())
    == set(EXPECTED_WINDOWS)
)

future_checks["observation_ids_unique"] = (
    future_population["observation_id"].nunique()
    == len(future_population)
)

future_checks["no_duplicate_observation_ids"] = (
    future_population["observation_id"].duplicated().sum()
    == 0
)
# Per-window audit
window_audit_rows = []

all_window_counts_match = True
all_window_label_counts_match = True
all_window_boundaries_match = True
all_window_period_types_match = True

for window in EXPECTED_WINDOWS:

    spec = future_spec_map[window]

    observed = (
        future_population.loc[
            future_population["period"] == window
        ]
        .copy()
    )

    expected_n = spec["n_observations"]

    observed_n = len(observed)

    observed_label_counts = (
        observed["sentiment"]
        .value_counts()
        .reindex(
            EXPECTED_LABEL_ORDER,
            fill_value=0
        )
        .astype(int)
        .to_dict()
    )

    expected_labels = expected_window_label_counts[window]

    observed_start = observed["date"].min()
    observed_end = observed["date"].max()

    expected_start = pd.Timestamp(
        spec["start_timestamp_utc"]
    )

    expected_end = pd.Timestamp(
        spec["end_timestamp_utc"]
    )

    expected_period_type = (
        "future_partial"
        if spec["is_partial_month"]
        else "future_month"
    )

    observed_period_types = (
        observed["period_type"]
        .dropna()
        .unique()
        .tolist()
    )

    count_match = (
        observed_n == expected_n
    )

    label_count_match = (
        observed_label_counts == expected_labels
    )

    boundary_match = (
        observed_start == expected_start
        and observed_end == expected_end
    )

    period_type_match = (
        observed_period_types
        == [expected_period_type]
    )

    all_window_counts_match &= count_match
    all_window_label_counts_match &= label_count_match
    all_window_boundaries_match &= boundary_match
    all_window_period_types_match &= period_type_match

    window_audit_rows.append({
        "period": window,
        "period_type": expected_period_type,
        "expected_n": int(expected_n),
        "observed_n": int(observed_n),
        "count_match": bool(count_match),

        "expected_negative_n": int(
            expected_labels["negative"]
        ),
        "observed_negative_n": int(
            observed_label_counts["negative"]
        ),

        "expected_neutral_n": int(
            expected_labels["neutral"]
        ),
        "observed_neutral_n": int(
            observed_label_counts["neutral"]
        ),

        "expected_positive_n": int(
            expected_labels["positive"]
        ),
        "observed_positive_n": int(
            observed_label_counts["positive"]
        ),

        "label_counts_match": bool(
            label_count_match
        ),

        "expected_start_utc": (
            expected_start.isoformat()
        ),
        "observed_start_utc": (
            observed_start.isoformat()
        ),

        "expected_end_utc": (
            expected_end.isoformat()
        ),
        "observed_end_utc": (
            observed_end.isoformat()
        ),

        "boundary_match": bool(
            boundary_match
        ),

        "period_type_match": bool(
            period_type_match
        ),
    })

window_audit_df = pd.DataFrame(
    window_audit_rows
)

future_checks["all_window_counts_match"] = (
    all_window_counts_match
)

future_checks["all_window_label_counts_match"] = (
    all_window_label_counts_match
)

future_checks["all_window_boundaries_match"] = (
    all_window_boundaries_match
)

future_checks["all_window_period_types_match"] = (
    all_window_period_types_match
)

# Source-composition cross-check
observed_source_counts = (
    future_population
    .groupby(
        ["period", "channel_username"],
        observed=True
    )
    .size()
    .reset_index(name="observed_count")
)

expected_source_counts = (
    frozen_future_source_manifest[
        [
            "split",
            "channel_username",
            "channel_count"
        ]
    ]
    .rename(
        columns={
            "split": "period",
            "channel_count": "expected_count"
        }
    )
    .copy()
)

source_crosscheck = (
    expected_source_counts
    .merge(
        observed_source_counts,
        on=[
            "period",
            "channel_username"
        ],
        how="outer"
    )
    .fillna({
        "expected_count": 0,
        "observed_count": 0
    })
)

source_crosscheck[
    "expected_count"
] = source_crosscheck[
    "expected_count"
].astype(int)

source_crosscheck[
    "observed_count"
] = source_crosscheck[
    "observed_count"
].astype(int)

source_crosscheck[
    "count_match"
] = (
    source_crosscheck["expected_count"]
    == source_crosscheck["observed_count"]
)

all_source_counts_match = bool(
    source_crosscheck["count_match"].all()
)

future_checks[
    "all_source_counts_match_day14"
] = all_source_counts_match

# Validate full period coverage with no overlap

assigned_count = (
    future_population["period"]
    .notna()
    .sum()
)

future_checks[
    "every_future_row_assigned_once"
] = (
    assigned_count == len(future_population)
)

# Overall Stage B3 result
all_future_checks_pass = all(
    future_checks.values()
)

# Console report

print("\n[FUTURE POPULATION CHECKS]")

for name, passed in future_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

print("\n" + "-" * 108)

print("\n[WINDOW AUDIT]")
display(
    window_audit_df[
        [
            "period",
            "period_type",
            "expected_n",
            "observed_n",
            "count_match",
            "label_counts_match",
            "boundary_match",
            "period_type_match",
        ]
    ]
)

print("\n[SOURCE CROSS-CHECK SUMMARY]")

source_mismatches = source_crosscheck.loc[
    ~source_crosscheck["count_match"]
].copy()

print(
    f"Source rows checked : "
    f"{len(source_crosscheck):,}"
)

print(
    f"Source mismatches   : "
    f"{len(source_mismatches):,}"
)

if len(source_mismatches) > 0:
    display(source_mismatches)

print("-" * 108)

print(
    f"Future n            : "
    f"{len(future_population):,}"
)

print(
    f"Future start UTC    : "
    f"{future_population['date'].min()}"
)

print(
    f"Future end UTC      : "
    f"{future_population['date'].max()}"
)

print(
    f"Unique obs IDs      : "
    f"{future_population['observation_id'].nunique():,}"
)

print("\nPer-window counts:")

for window in EXPECTED_WINDOWS:
    n_window = int(
        (future_population["period"] == window).sum()
    )
    print(
        f"  {window}: {n_window:,}"
    )

print("-" * 108)

# Write permanent future_population_audit.json on PASS

future_population_audit = {
    "day": 16,
    "artifact": "future_population_audit",
    "project": (
        "Financial Language Models under "
        "Temporal Distribution Shift"
    ),

    "benchmark_population": {
        "verified_n": int(
            len(benchmark_population)
        ),
        "source_artifact": str(
            DATASET_B_SOURCE
        ),
    },

    "future_population": {
        "expected_n": int(
            expected_future_n
        ),
        "observed_n": int(
            len(future_population)
        ),
        "cutoff_utc": (
            cutoff_utc.isoformat()
        ),
        "observed_start_utc": (
            future_population[
                "date"
            ].min().isoformat()
        ),
        "observed_end_utc": (
            future_population[
                "date"
            ].max().isoformat()
        ),
        "unique_observation_ids": int(
            future_population[
                "observation_id"
            ].nunique()
        ),
    },

    "window_audit": (
        window_audit_df
        .to_dict(orient="records")
    ),

    "source_crosscheck": {
        "rows_checked": int(
            len(source_crosscheck)
        ),
        "mismatch_count": int(
            len(source_mismatches)
        ),
        "all_counts_match": bool(
            all_source_counts_match
        ),
    },

    "checks": {
        key: bool(value)
        for key, value
        in future_checks.items()
    },

    "all_checks_pass": bool(
        all_future_checks_pass
    ),

    "protocol_status": (
        "PASS"
        if all_future_checks_pass
        else "FAIL"
    ),

    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}

future_population_audit_path = (
    DAY16_DIR
    / "future_population_audit.json"
)

if all_future_checks_pass:

    with open(
        future_population_audit_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            future_population_audit,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        "STAGE B3 STATUS : PASS — "
        "EXACT F1–F6 FUTURE POPULATION VERIFIED"
    )

    print(
        f"Audit saved         : "
        f"{future_population_audit_path}"
    )

else:

    print(
        "STAGE B3 STATUS : FAIL — "
        "STOP BEFORE MODEL INFERENCE"
    )

print("=" * 108)

if not all_future_checks_pass:

    failed_checks = [
        name
        for name, passed
        in future_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage B3 future-population "
        "audit failed:\n- "
        + "\n- ".join(
            failed_checks
        )
    )

## Stage C — Prepare the Fixed Inference Pipeline

### Stage C1 — Frozen Model and Runtime Preflight

Before loading the model into GPU memory, verify that the current runtime can reproduce the frozen Day15 inference configuration.

This preflight checks:

- required model libraries are available;
- the frozen adapter directory is intact;
- PEFT adapter configuration is readable;
- the adapter points to the frozen Meta-Llama-3-8B base model;
- Day14 and Day15 inference settings remain consistent;
- no training or adaptation state is activated.

No future prediction is generated in this step.

The Day16 inference pipeline remains fixed as:

- model: frozen historical model `M_H`;
- base model: `meta-llama/Meta-Llama-3-8B`;
- frozen Day15 LoRA adapter;
- 4-bit NF4 quantization;
- evaluation mode;
- left padding;
- `do_sample = False`;
- `max_new_tokens = 4`;
- exact-label decoding only;
- valid predictions: `negative`, `neutral`, `positive`.

In [ ]:
%pip install -q bitsandbytes

import bitsandbytes as bnb

print("=" * 88)
print("DAY 16 — BITSANDBYTES RUNTIME RECOVERY")
print("=" * 88)
print(f"bitsandbytes version : {bnb.__version__}")
print("Import status        : PASS")
print("=" * 88)

In [ ]:
# Stage C1 frozen model and runtime preflight
import importlib
from pathlib import Path

print("=" * 108)
print("DAY 16 — STAGE C1: FROZEN MODEL / RUNTIME PREFLIGHT")
print("=" * 108)

# Required libraries
required_libraries = [
    "transformers",
    "peft",
    "bitsandbytes",
    "accelerate",
]

library_versions = {}
library_checks = {}

for lib_name in required_libraries:
    try:
        module = importlib.import_module(lib_name)

        version = getattr(
            module,
            "__version__",
            "unknown"
        )

        library_versions[lib_name] = version
        library_checks[f"{lib_name}_available"] = True

    except Exception as e:
        library_versions[lib_name] = f"ERROR: {e}"
        library_checks[f"{lib_name}_available"] = False
# Frozen adapter artifact inspection
adapter_required_files = [
    "adapter_config.json",
]

adapter_file_checks = {
    filename: (
        FROZEN_ADAPTER_PATH / filename
    ).exists()
    for filename in adapter_required_files
}

# Adapter weights can be either safetensors or bin
adapter_weight_candidates = [
    FROZEN_ADAPTER_PATH / "adapter_model.safetensors",
    FROZEN_ADAPTER_PATH / "adapter_model.bin",
]

adapter_weights_found = [
    p for p in adapter_weight_candidates
    if p.exists()
]

# Read adapter config
adapter_config_path = (
    FROZEN_ADAPTER_PATH / "adapter_config.json"
)

if not adapter_config_path.exists():
    raise FileNotFoundError(
        f"Missing adapter config: {adapter_config_path}"
    )

with open(
    adapter_config_path,
    "r",
    encoding="utf-8"
) as f:
    adapter_config = load_artifact_json(f)
# Expected frozen identities
expected_base_model = frozen_model_manifest[
    "model_identity"
]["base_model"]

adapter_base_model = adapter_config.get(
    "base_model_name_or_path"
)

# Some PEFT versions may retain equivalent spelling/path forms.
base_model_identity_match = (
    str(adapter_base_model).rstrip("/")
    == str(expected_base_model).rstrip("/")
)

# Protocol consistency checks
preflight_checks = {}

preflight_checks.update(library_checks)

preflight_checks["adapter_directory_exists"] = (
    FROZEN_ADAPTER_PATH.exists()
)

preflight_checks["adapter_config_exists"] = (
    adapter_config_path.exists()
)

preflight_checks["adapter_weights_exist"] = (
    len(adapter_weights_found) >= 1
)

preflight_checks["base_model_identity_matches"] = (
    base_model_identity_match
)

preflight_checks["frozen_model_is_M_H"] = (
    frozen_model_manifest[
        "model_identity"
    ]["model_id"] == "M_H"
)

preflight_checks["frozen_status_valid"] = (
    frozen_model_manifest[
        "model_identity"
    ]["freeze_status"]
    == "FROZEN_BEFORE_FUTURE_EVALUATION"
)

preflight_checks["quantization_enabled"] = (
    frozen_model_manifest[
        "quantization"
    ]["enabled"] is True
)

preflight_checks["load_in_4bit"] = (
    frozen_model_manifest[
        "quantization"
    ]["load_in_4bit"] is True
)

preflight_checks["quant_type_nf4"] = (
    frozen_model_manifest[
        "quantization"
    ]["bnb_4bit_quant_type"] == "nf4"
)

preflight_checks["double_quant_enabled"] = (
    frozen_model_manifest[
        "quantization"
    ]["bnb_4bit_use_double_quant"] is True
)

preflight_checks["inference_eval_mode"] = (
    frozen_model_manifest[
        "inference_protocol"
    ]["model_mode"] == "eval"
)

preflight_checks["left_padding"] = (
    frozen_model_manifest[
        "inference_protocol"
    ]["padding_side"] == "left"
)

preflight_checks["deterministic_generation"] = (
    frozen_model_manifest[
        "inference_protocol"
    ]["do_sample"] is False
)

preflight_checks["max_new_tokens_is_4"] = (
    frozen_model_manifest[
        "inference_protocol"
    ]["max_new_tokens"] == 4
)

preflight_checks["valid_prediction_labels_match"] = (
    frozen_model_manifest[
        "inference_protocol"
    ]["valid_predictions"]
    == EXPECTED_LABEL_ORDER
)

print("\n[LIBRARY VERSIONS]")

for name, version in library_versions.items():
    print(f"{name:<15}: {version}")

print("\n" + "-" * 108)

print("\n[FROZEN ADAPTER]")

print(
    f"Adapter directory   : "
    f"{FROZEN_ADAPTER_PATH}"
)

print(
    f"Adapter config      : "
    f"{adapter_config_path.exists()}"
)

print(
    "Adapter weights     : "
    + (
        ", ".join(
            p.name
            for p in adapter_weights_found
        )
        if adapter_weights_found
        else "NOT FOUND"
    )
)

print(
    f"Manifest base model : "
    f"{expected_base_model}"
)

print(
    f"Adapter base model  : "
    f"{adapter_base_model}"
)

print("\n" + "-" * 108)

print("\n[PREFLIGHT CHECKS]")

for name, passed in preflight_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

print("-" * 108)

all_preflight_checks_pass = all(
    preflight_checks.values()
)

if all_preflight_checks_pass:
    print(
        "STAGE C1 STATUS : PASS — "
        "FROZEN MODEL RUNTIME PREFLIGHT VERIFIED"
    )
else:
    print(
        "STAGE C1 STATUS : FAIL — "
        "STOP BEFORE MODEL LOADING"
    )

print("=" * 108)

if not all_preflight_checks_pass:

    failed_checks = [
        name
        for name, passed
        in preflight_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage C1 preflight failed:\n- "
        + "\n- ".join(failed_checks)
    )

### Stage C2 — Load the Frozen Tokenizer, Base Model, and LoRA Adapter

Load the exact frozen Day15 inference stack:

- tokenizer from the frozen Meta-Llama-3-8B base model;
- base model in 4-bit NF4 quantization;
- frozen Day15 LoRA adapter;
- left-padding inference configuration;
- evaluation mode with gradients disabled.

This step verifies that:

- the tokenizer loads successfully;
- the base model is loaded with the frozen quantization protocol;
- the frozen adapter attaches successfully;
- no parameters remain trainable;
- the loaded model is ready for deterministic Day16 inference.

No future observation is evaluated in this cell.

In [ ]:
# Load frozen tokenizer + 4-bit base model + frozen LoRA adapter
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel

print("=" * 108)
print("DAY 16 — STAGE C2: LOAD FROZEN INFERENCE STACK")
print("=" * 108)

# Frozen model identity
BASE_MODEL_NAME = frozen_model_manifest[
    "model_identity"
]["base_model"]

MAX_SEQ_LENGTH = frozen_model_manifest[
    "sequence_protocol"
]["max_seq_length"]

# Tokenizer
print("\n[1/4] Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True,
)

# Frozen inference protocol
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
# Frozen 4-bit quantization config
print("\n[2/4] Building frozen 4-bit quantization config...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Quantization config ready.")

# Load base model
print("\n[3/4] Loading 4-bit base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Base model loaded.")

# Attach frozen LoRA adapter
print("\n[4/4] Loading frozen Day15 LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    str(FROZEN_ADAPTER_PATH),
    is_trainable=False,
)

model.eval()

# Explicitly disable gradients
for param in model.parameters():
    param.requires_grad = False

# Integrity checks
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

model_device = next(
    model.parameters()
).device

load_checks = {
    "tokenizer_loaded": tokenizer is not None,
    "pad_token_defined": tokenizer.pad_token is not None,
    "padding_side_left": tokenizer.padding_side == "left",
    "model_loaded": model is not None,
    "model_in_eval_mode": model.training is False,
    "trainable_parameters_zero": trainable_params == 0,
    "frozen_adapter_path_exists": FROZEN_ADAPTER_PATH.exists(),
}

print("\n" + "-" * 108)
print("[LOADED INFERENCE STACK]")

print(f"Base model            : {BASE_MODEL_NAME}")
print(f"Frozen adapter        : {FROZEN_ADAPTER_PATH}")
print(f"Tokenizer pad token   : {repr(tokenizer.pad_token)}")
print(f"Tokenizer padding side: {tokenizer.padding_side}")
print(f"Max sequence length   : {MAX_SEQ_LENGTH}")
print(f"Model device          : {model_device}")
print(f"Total parameters      : {total_params:,}")
print(f"Trainable parameters  : {trainable_params:,}")

print("\n" + "-" * 108)
print("[LOAD CHECKS]")

for name, passed in load_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

print("-" * 108)

all_load_checks_pass = all(
    load_checks.values()
)

if all_load_checks_pass:
    print(
        "STAGE C2 STATUS : PASS — "
        "FROZEN INFERENCE STACK LOADED"
    )
else:
    print(
        "STAGE C2 STATUS : FAIL — "
        "STOP BEFORE TOKENIZATION OR INFERENCE"
    )

print("=" * 108)

if not all_load_checks_pass:
    failed_checks = [
        name
        for name, passed
        in load_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage C2 model-loading failure:\n- "
        + "\n- ".join(failed_checks)
    )

### Stage C3 — Freeze the Exact Day16 Prompt and Decoder

Reconstruct the Day16 inference pipeline directly from the Day15 training and historical-validation implementation.

The exact frozen instruction is:

> `Classify the sentiment of the following financial news as negative, neutral, or positive.`

The prompt is constructed with the Day14/Day15 frozen template:

`Instruction:{instruction}\nInput:{input}\nAnswer: `

Importantly, the stored `\n` sequences are preserved exactly as **literal backslash-n characters**, matching the frozen Day15 implementation.

The inference procedure also inherits the Day15 settings:

- tokenizer padding side: left;
- pad token: EOS token;
- maximum prompt length: 512 tokens;
- no added special tokens;
- greedy generation (`do_sample=False`);
- `max_new_tokens=4`;
- decode generated tokens only;
- strip leading/trailing whitespace;
- lowercase output;
- exact valid labels only:
  - `negative`
  - `neutral`
  - `positive`

Before evaluating any future observation, this cell performs a small **historical-validation reproducibility check** against predictions already saved on Day15.

No F1–F6 prediction is generated in this stage.

In [ ]:
# Freeze exact prompt builder / decoder
# + historical-validation reproducibility sanity check
from tqdm.auto import tqdm

print("=" * 112)
print("DAY 16 — STAGE C3: FREEZE PROMPT / DECODER + HISTORICAL REPRODUCIBILITY CHECK")
print("=" * 112)

# Exact frozen inference constants recovered from Day15
PROMPT_TEMPLATE = frozen_model_manifest[
    "sequence_protocol"
]["prompt_template"]

INSTRUCTION_TEXT = (
    "Classify the sentiment of the following financial news "
    "as negative, neutral, or positive."
)

VALID_PREDICTIONS = set(
    frozen_model_manifest[
        "inference_protocol"
    ]["valid_predictions"]
)

MAX_SEQ_LENGTH = frozen_model_manifest[
    "sequence_protocol"
]["max_seq_length"]

INFERENCE_BATCH_SIZE = 4

MAX_NEW_TOKENS = frozen_model_manifest[
    "inference_protocol"
]["max_new_tokens"]

DO_SAMPLE = frozen_model_manifest[
    "inference_protocol"
]["do_sample"]

# Frozen inference tokenizer convention
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

# Exact Day15-style prompt builder
def build_inference_prompt(text):
    """
    Reproduce the exact frozen Day15 inference prompt.

    IMPORTANT:
    PROMPT_TEMPLATE intentionally retains literal '\\n'
    sequences from the Day14/Day15 frozen protocol.
    """

    return PROMPT_TEMPLATE.format(
        instruction=INSTRUCTION_TEXT,
        input=str(text),
    )

# Exact Day15-style prediction decoder
def decode_generated_predictions(
    output_ids,
    prompt_width,
):
    """
    Decode generated tokens only, following the frozen
    Day15 inference protocol.
    """

    generated_ids = output_ids[
        :,
        prompt_width:
    ]

    decoded = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return [
        x.strip().lower()
        for x in decoded
    ]

# Frozen prompt / tokenizer checks
prompt_checks = {
    "instruction_exact":
        INSTRUCTION_TEXT
        ==
        (
            "Classify the sentiment of the following financial news "
            "as negative, neutral, or positive."
        ),

    "literal_backslash_n_preserved":
        "\\n" in PROMPT_TEMPLATE,

    "max_seq_length_512":
        MAX_SEQ_LENGTH == 512,

    "padding_side_left":
        tokenizer.padding_side == "left",

    "pad_token_is_eos":
        tokenizer.pad_token_id
        == tokenizer.eos_token_id,

    "batch_size_matches_day15":
        INFERENCE_BATCH_SIZE == 4,

    "max_new_tokens_4":
        MAX_NEW_TOKENS == 4,

    "do_sample_false":
        DO_SAMPLE is False,

    "valid_labels_exact":
        VALID_PREDICTIONS
        == {
            "negative",
            "neutral",
            "positive",
        },
}

print("\n[PROMPT / DECODER PROTOCOL CHECKS]")

for name, passed in prompt_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

if not all(prompt_checks.values()):
    raise RuntimeError(
        "Frozen prompt/decoder protocol check failed."
    )
# Load existing Day15 validation predictions
day15_validation_predictions = pd.read_csv(
    DAY15_DIR / "historical_validation_predictions.csv"
)

# Use a small deterministic historical sample only.
# No future rows are touched.
SANITY_N = 16

sanity_reference = (
    day15_validation_predictions
    .iloc[:SANITY_N]
    .copy()
)

# Recover text by observation_id from the already-verified
# benchmark population.
historical_text_lookup = (
    benchmark_population.loc[
        benchmark_population["date"] < cutoff_utc,
        [
            "observation_id",
            "text",
        ]
    ]
    .copy()
)

sanity_df = sanity_reference.merge(
    historical_text_lookup,
    on="observation_id",
    how="left",
    validate="one_to_one",
)

if sanity_df["text"].isna().any():
    raise RuntimeError(
        "Historical sanity-check text recovery failed."
    )

# Re-run frozen M_H on the historical sample
current_predictions = []

model.eval()

for start_idx in range(
    0,
    len(sanity_df),
    INFERENCE_BATCH_SIZE,
):

    end_idx = min(
        start_idx + INFERENCE_BATCH_SIZE,
        len(sanity_df),
    )

    batch_df = sanity_df.iloc[
        start_idx:end_idx
    ]

    prompts = [
        build_inference_prompt(text)
        for text in batch_df["text"]
    ]

    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )

    encoded = {
        k: v.to(model.device)
        for k, v in encoded.items()
    }

    prompt_width = encoded[
        "input_ids"
    ].shape[1]

    with torch.no_grad():

        output_ids = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = decode_generated_predictions(
        output_ids=output_ids,
        prompt_width=prompt_width,
    )

    current_predictions.extend(decoded)

# Compare against Day15 frozen predictions
sanity_df[
    "day16_reproduced_prediction"
] = current_predictions

sanity_df[
    "prediction_match"
] = (
    sanity_df[
        "predicted_label"
    ].astype(str)
    ==
    sanity_df[
        "day16_reproduced_prediction"
    ].astype(str)
)

sanity_df[
    "day16_prediction_valid"
] = sanity_df[
    "day16_reproduced_prediction"
].isin(
    VALID_PREDICTIONS
)

repro_checks = {
    "sanity_n_16":
        len(sanity_df) == SANITY_N,

    "all_reference_predictions_present":
        sanity_df[
            "predicted_label"
        ].notna().all(),

    "all_reproduced_predictions_present":
        sanity_df[
            "day16_reproduced_prediction"
        ].notna().all(),

    "all_reproduced_predictions_valid":
        sanity_df[
            "day16_prediction_valid"
        ].all(),

    "all_predictions_match_day15":
        sanity_df[
            "prediction_match"
        ].all(),
}

# Report
print("\n" + "-" * 112)
print("[HISTORICAL VALIDATION REPRODUCIBILITY CHECK]")

for name, passed in repro_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

print("\nComparison:")

display(
    sanity_df[
        [
            "observation_id",
            "true_label",
            "predicted_label",
            "day16_reproduced_prediction",
            "prediction_match",
        ]
    ]
)

print("-" * 112)

all_stage_c3_checks_pass = (
    all(prompt_checks.values())
    and all(repro_checks.values())
)

if all_stage_c3_checks_pass:

    print(
        "STAGE C3 STATUS : PASS — "
        "DAY15 INFERENCE PIPELINE REPRODUCED"
    )

else:

    print(
        "STAGE C3 STATUS : FAIL — "
        "STOP BEFORE FUTURE INFERENCE"
    )

print("=" * 112)

if not all_stage_c3_checks_pass:

    failed = [
        name
        for name, passed
        in {
            **prompt_checks,
            **repro_checks,
        }.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage C3 reproducibility check failed:\n- "
        + "\n- ".join(failed)
    )

## Stage D — Strict Future Inference

### Stage D1 — Run the Frozen Model on All 20,002 Future Observations

Apply the fully reproduced frozen historical model `M_H` to the exact Day14-defined F1–F6 future population.

The inference procedure is identical for every future observation:

- one frozen model;
- one frozen prompt;
- one tokenizer configuration;
- one decoding rule;
- no retraining;
- no calibration;
- no threshold tuning;
- no reweighting;
- no resampling;
- no adaptation by temporal window.

Future-window labels are carried only as metadata and never affect model inference.

The inference loop supports recovery from an interrupted Colab session through a temporary prefix-validated partial prediction file. The partial file is not a final Day16 artifact and will be removed after successful completion.

The final output of this stage is:

`future_predictions.csv`

This observation-level file becomes the authoritative Day16 evidence layer for all subsequent aggregate metrics and Day17 failure-mode diagnosis.

In [ ]:
# Run frozen M_H inference on exact 20,002 future observations
import time
import pandas as pd
import torch
import transformers

from tqdm.auto import tqdm

print("=" * 116)
print("DAY 16 — STAGE D1: STRICT FUTURE INFERENCE")
print("=" * 116)

# Final frozen-inference assertions
assert all_future_checks_pass, (
    "Future-population audit has not passed."
)

assert all_stage_c3_checks_pass, (
    "Day15 inference reproducibility check has not passed."
)

assert model.training is False, (
    "Frozen model is not in evaluation mode."
)

assert sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
) == 0, (
    "Frozen model unexpectedly contains trainable parameters."
)
# Prepare exact immutable inference population
future_inference_df = (
    future_population[
        [
            "observation_id",
            "channel_username",
            "message_id",
            "date",
            "period",
            "period_type",
            "text",
            "sentiment",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

future_inference_df["date"] = (
    future_inference_df["date"]
    .astype(str)
)

# Formal population checks immediately before inference
assert len(future_inference_df) == 20_002

assert (
    future_inference_df["observation_id"].nunique()
    == 20_002
)

assert future_inference_df["period"].notna().all()

assert set(
    future_inference_df["period"].unique()
) == set(EXPECTED_WINDOWS)

# Output / recovery paths
partial_prediction_path = (
    DAY16_DIR
    / "future_predictions_partial.csv"
)

final_prediction_path = (
    DAY16_DIR
    / "future_predictions.csv"
)

# Recovery logic

if partial_prediction_path.exists():

    partial_df = pd.read_csv(
        partial_prediction_path
    )

    completed_n = len(partial_df)

    if completed_n > len(future_inference_df):
        raise RuntimeError(
            "Partial prediction file contains more rows "
            "than the frozen future population."
        )

    required_partial_columns = {
        "observation_id",
        "timestamp_utc",
        "period",
        "period_type",
        "source",
        "message_id",
        "text",
        "y_true",
        "y_pred",
        "prediction_valid",
        "correct",
        "transition",
        "inference_status",
    }

    missing_partial_columns = (
        required_partial_columns
        - set(partial_df.columns)
    )

    if missing_partial_columns:
        raise RuntimeError(
            "Partial prediction file has an incompatible schema: "
            + ", ".join(sorted(missing_partial_columns))
        )

    # Recovery is allowed only if the partial file is the exact
    # prefix of the frozen 20,002-observation population.
    expected_ids = (
        future_inference_df[
            "observation_id"
        ]
        .iloc[:completed_n]
        .astype(str)
        .tolist()
    )

    actual_ids = (
        partial_df[
            "observation_id"
        ]
        .astype(str)
        .tolist()
    )

    if actual_ids != expected_ids:
        raise RuntimeError(
            "Partial future prediction file does not match "
            "the frozen population ordering."
        )

    prediction_records = (
        partial_df
        .to_dict("records")
    )

    print(
        f"Recovery file found  : YES"
    )
    print(
        f"Completed rows       : {completed_n:,}"
    )

else:

    completed_n = 0
    prediction_records = []

    print(
        "Recovery file found  : NO"
    )
    print(
        "Completed rows       : 0"
    )
# Freeze runtime inference settings

model.eval()

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

assert INFERENCE_BATCH_SIZE == 4
assert MAX_SEQ_LENGTH == 512
assert MAX_NEW_TOKENS == 4
assert DO_SAMPLE is False

print("-" * 116)
print(f"Future population    : {len(future_inference_df):,}")
print(f"Starting at row      : {completed_n:,}")
print(f"Inference batch size : {INFERENCE_BATCH_SIZE}")
print(f"Max sequence length  : {MAX_SEQ_LENGTH}")
print(f"Max new tokens       : {MAX_NEW_TOKENS}")
print(f"Sampling             : {DO_SAMPLE}")
print("-" * 116)

# Run one fixed inference pipeline over all remaining rows
inference_start_time = time.time()

# Suppress only the repetitive Transformers
# max_length/max_new_tokens logger message.
# This does NOT alter generation behavior.
previous_verbosity = (
    transformers.logging.get_verbosity()
)

transformers.logging.set_verbosity_error()

try:

    for start_idx in tqdm(
        range(
            completed_n,
            len(future_inference_df),
            INFERENCE_BATCH_SIZE,
        ),
        desc="Day16 Future Inference",
    ):

        end_idx = min(
            start_idx + INFERENCE_BATCH_SIZE,
            len(future_inference_df),
        )

        batch_df = future_inference_df.iloc[
            start_idx:end_idx
        ]

        prompts = [
            build_inference_prompt(text)
            for text in batch_df["text"]
        ]

        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            add_special_tokens=False,
        )

        encoded = {
            key: value.to(model.device)
            for key, value in encoded.items()
        }

        prompt_width = encoded[
            "input_ids"
        ].shape[1]

        with torch.no_grad():

            output_ids = model.generate(
                **encoded,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=DO_SAMPLE,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        decoded_predictions = (
            decode_generated_predictions(
                output_ids=output_ids,
                prompt_width=prompt_width,
            )
        )

        if len(decoded_predictions) != len(batch_df):
            raise RuntimeError(
                "Generated prediction count does not match "
                "the current inference batch."
            )
        # Observation-level immutable evidence records
        for local_i, (_, row) in enumerate(
            batch_df.iterrows()
        ):

            prediction = str(
                decoded_predictions[local_i]
            )

            true_label = str(
                row["sentiment"]
            ).strip().lower()

            prediction_valid = (
                prediction in VALID_PREDICTIONS
            )

            # Frozen Day14 rule:
            # invalid predictions count as errors.
            correct = int(
                prediction_valid
                and prediction == true_label
            )

            transition = (
                f"{true_label}->{prediction}"
                if prediction_valid
                else f"{true_label}->invalid"
            )

            inference_status = (
                "success"
                if prediction_valid
                else "invalid_prediction"
            )

            prediction_records.append(
                {
                    "observation_id":
                        row["observation_id"],

                    "timestamp_utc":
                        row["date"],

                    "period":
                        row["period"],

                    "period_type":
                        row["period_type"],

                    "source":
                        row["channel_username"],

                    "message_id":
                        row["message_id"],

                    "text":
                        row["text"],

                    "y_true":
                        true_label,

                    "y_pred":
                        prediction,

                    "prediction_valid":
                        bool(prediction_valid),

                    "correct":
                        int(correct),

                    "transition":
                        transition,

                    "inference_status":
                        inference_status,
                }
            )
        # Periodic recovery save

        # Save approximately every 200 completed observations,
        # matching the recovery philosophy used on Day15.
        if (
            len(prediction_records) % 200
            < INFERENCE_BATCH_SIZE
            or end_idx == len(future_inference_df)
        ):

            pd.DataFrame(
                prediction_records
            ).to_csv(
                partial_prediction_path,
                index=False,
            )

finally:

    # Restore normal Transformers logging
    transformers.logging.set_verbosity(
        previous_verbosity
    )
# Finalize prediction table
future_predictions = pd.DataFrame(
    prediction_records
)

inference_elapsed_seconds = (
    time.time()
    - inference_start_time
)
# Formal prediction-integrity checks
prediction_checks = {
    "prediction_n_20002":
        len(future_predictions) == 20_002,

    "observation_ids_unique":
        future_predictions[
            "observation_id"
        ].nunique() == 20_002,

    "observation_order_preserved":
        future_predictions[
            "observation_id"
        ].astype(str).tolist()
        ==
        future_inference_df[
            "observation_id"
        ].astype(str).tolist(),

    "all_periods_present":
        set(
            future_predictions[
                "period"
            ].unique()
        )
        == set(EXPECTED_WINDOWS),

    "period_order_preserved":
        future_predictions[
            "period"
        ].astype(str).tolist()
        ==
        future_inference_df[
            "period"
        ].astype(str).tolist(),

    "all_true_labels_valid":
        set(
            future_predictions[
                "y_true"
            ].unique()
        )
        == set(EXPECTED_LABEL_ORDER),

    "all_predictions_nonmissing":
        future_predictions[
            "y_pred"
        ].notna().all(),

    "every_row_has_inference_status":
        future_predictions[
            "inference_status"
        ].notna().all(),

    "every_row_has_transition":
        future_predictions[
            "transition"
        ].notna().all(),

    "correct_is_binary":
        set(
            future_predictions[
                "correct"
            ].unique()
        ).issubset({0, 1}),

    "future_dates_only":
        (
            pd.to_datetime(
                future_predictions[
                    "timestamp_utc"
                ],
                utc=True,
            )
            >= cutoff_utc
        ).all(),

    "model_still_eval":
        model.training is False,

    "trainable_parameters_still_zero":
        sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        ) == 0,
}

all_prediction_checks_pass = all(
    prediction_checks.values()
)

# Save final authoritative future_predictions.csv
if all_prediction_checks_pass:

    future_predictions.to_csv(
        final_prediction_path,
        index=False,
    )

    # Remove temporary recovery artifact only after the
    # finalized evidence layer has been successfully written.
    if partial_prediction_path.exists():
        partial_prediction_path.unlink()
# Compact console report
print("\n" + "-" * 116)
print("[FUTURE INFERENCE INTEGRITY CHECKS]")

for name, passed in prediction_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

print("-" * 116)

valid_prediction_n = int(
    future_predictions[
        "prediction_valid"
    ].sum()
)

invalid_prediction_n = int(
    (
        ~future_predictions[
            "prediction_valid"
        ]
    ).sum()
)

print(
    f"Predictions produced : "
    f"{len(future_predictions):,}"
)

print(
    f"Valid predictions    : "
    f"{valid_prediction_n:,}"
)

print(
    f"Invalid predictions  : "
    f"{invalid_prediction_n:,}"
)

print(
    f"Elapsed seconds      : "
    f"{inference_elapsed_seconds:,.2f}"
)

print(
    f"Final artifact       : "
    f"{final_prediction_path}"
)

print(
    f"Final file exists    : "
    f"{final_prediction_path.exists()}"
)

print(
    f"Partial file removed : "
    f"{not partial_prediction_path.exists()}"
)

print("-" * 116)

if all_prediction_checks_pass:

    print(
        "STAGE D1 STATUS : PASS — "
        "STRICT FUTURE INFERENCE COMPLETE"
    )

else:

    print(
        "STAGE D1 STATUS : FAIL — "
        "DO NOT COMPUTE FUTURE METRICS"
    )

print("=" * 116)

if not all_prediction_checks_pass:

    failed_checks = [
        name
        for name, passed
        in prediction_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 future-inference integrity failure:\n- "
        + "\n- ".join(failed_checks)
    )

## Stage E — Frozen Per-Window Performance Metrics

### Stage E1 — Compute the Pre-Specified Metrics for F1–F6

Compute the frozen Day14 evaluation endpoints separately for each strictly future window.

The endpoint hierarchy remains unchanged:

**Primary endpoint**
- Macro-F1

**Secondary endpoints**
- Accuracy
- Weighted-F1

**Class-specific performance**
- Negative recall
- Neutral recall
- Positive recall

All metrics are computed from the finalized observation-level artifact:

`future_predictions.csv`

using the frozen label order:

`negative → neutral → positive`

No temporal trend model, causal interpretation, smoothing, window regrouping, or endpoint reprioritization is performed in this stage.

The output artifact is:

`future_window_metrics.csv`

In [ ]:
# Stage E1: compute frozen per-window metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
)

print("=" * 112)
print("DAY 16 — STAGE E1: FROZEN PER-WINDOW PERFORMANCE METRICS")
print("=" * 112)

# ------------------------------------------------------------
# 1. Reload authoritative evidence layer
# ------------------------------------------------------------

future_predictions = pd.read_csv(
    DAY16_DIR / "future_predictions.csv"
)

LABEL_ORDER = [
    "negative",
    "neutral",
    "positive",
]

# Evidence-layer integrity check

metric_input_checks = {
    "prediction_n_20002":
        len(future_predictions) == 20_002,

    "periods_exactly_F1_to_F6":
        set(
            future_predictions["period"].unique()
        )
        == set(EXPECTED_WINDOWS),

    "all_true_labels_valid":
        set(
            future_predictions["y_true"].unique()
        )
        == set(LABEL_ORDER),

    "all_predictions_valid":
        set(
            future_predictions["y_pred"].unique()
        ).issubset(
            set(LABEL_ORDER)
        ),

    "no_missing_true_labels":
        future_predictions[
            "y_true"
        ].notna().all(),

    "no_missing_predictions":
        future_predictions[
            "y_pred"
        ].notna().all(),
}

print("\n[METRIC INPUT CHECKS]")

for name, passed in metric_input_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

if not all(metric_input_checks.values()):
    failed = [
        name
        for name, passed
        in metric_input_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Metric input integrity failure:\n- "
        + "\n- ".join(failed)
    )

# Compute frozen metrics window by window
window_metric_records = []

for period in EXPECTED_WINDOWS:

    period_df = (
        future_predictions.loc[
            future_predictions["period"] == period
        ]
        .copy()
    )

    y_true = (
        period_df["y_true"]
        .astype(str)
    )

    y_pred = (
        period_df["y_pred"]
        .astype(str)
    )

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average="weighted",
        zero_division=0,
    )

    class_recalls = recall_score(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average=None,
        zero_division=0,
    )

    period_type_values = (
        period_df["period_type"]
        .dropna()
        .unique()
        .tolist()
    )

    if len(period_type_values) != 1:
        raise RuntimeError(
            f"{period} has inconsistent period_type values."
        )

    window_metric_records.append(
        {
            "period":
                period,

            "period_type":
                period_type_values[0],

            "n":
                int(len(period_df)),

            "macro_f1":
                float(macro_f1),

            "accuracy":
                float(accuracy),

            "weighted_f1":
                float(weighted_f1),

            "negative_recall":
                float(class_recalls[0]),

            "neutral_recall":
                float(class_recalls[1]),

            "positive_recall":
                float(class_recalls[2]),
        }
    )

future_window_metrics = pd.DataFrame(
    window_metric_records
)

# Add frozen temporal boundaries from Day14
window_boundary_table = (
    future_window_manifest[
        [
            "future_window",
            "start_timestamp_utc",
            "end_timestamp_utc",
        ]
    ]
    .rename(
        columns={
            "future_window": "period",
            "start_timestamp_utc": "start_utc",
            "end_timestamp_utc": "end_utc",
        }
    )
)

future_window_metrics = (
    future_window_metrics
    .merge(
        window_boundary_table,
        on="period",
        how="left",
        validate="one_to_one",
    )
)

# Reorder columns
future_window_metrics = future_window_metrics[
    [
        "period",
        "period_type",
        "start_utc",
        "end_utc",
        "n",
        "macro_f1",
        "accuracy",
        "weighted_f1",
        "negative_recall",
        "neutral_recall",
        "positive_recall",
    ]
]

# Frozen count cross-check
expected_counts = {
    row["future_window"]: int(row["n"])
    for _, row in future_window_manifest.iterrows()
}

metric_checks = {
    "six_future_windows":
        len(future_window_metrics) == 6,

    "window_order_preserved":
        future_window_metrics[
            "period"
        ].tolist()
        == EXPECTED_WINDOWS,

    "window_counts_match_day14":
        all(
            int(
                future_window_metrics.loc[
                    future_window_metrics["period"] == period,
                    "n",
                ].iloc[0]
            )
            == expected_counts[period]
            for period in EXPECTED_WINDOWS
        ),

    "macro_f1_valid_range":
        future_window_metrics[
            "macro_f1"
        ].between(0, 1).all(),

    "accuracy_valid_range":
        future_window_metrics[
            "accuracy"
        ].between(0, 1).all(),

    "weighted_f1_valid_range":
        future_window_metrics[
            "weighted_f1"
        ].between(0, 1).all(),

    "all_class_recalls_valid":
        future_window_metrics[
            [
                "negative_recall",
                "neutral_recall",
                "positive_recall",
            ]
        ]
        .apply(
            lambda s: s.between(0, 1).all()
        )
        .all(),
}

# Save formal artifact
future_window_metrics_path = (
    DAY16_DIR
    / "future_window_metrics.csv"
)

future_window_metrics.to_csv(
    future_window_metrics_path,
    index=False,
)

metric_checks["artifact_saved"] = (
    future_window_metrics_path.exists()
)

# Report

print("\n" + "-" * 112)
print("[FROZEN WINDOW METRICS]")

display(
    future_window_metrics[
        [
            "period",
            "period_type",
            "n",
            "macro_f1",
            "accuracy",
            "weighted_f1",
            "negative_recall",
            "neutral_recall",
            "positive_recall",
        ]
    ]
)

print("\n" + "-" * 112)
print("[STAGE E1 CHECKS]")

for name, passed in metric_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

all_metric_checks_pass = all(
    metric_checks.values()
)

print("-" * 112)

print(
    f"Artifact saved      : "
    f"{future_window_metrics_path}"
)

if all_metric_checks_pass:

    print(
        "STAGE E1 STATUS : PASS — "
        "FROZEN PER-WINDOW METRICS COMPUTED"
    )

else:

    print(
        "STAGE E1 STATUS : FAIL — "
        "STOP BEFORE CLASS-LEVEL ANALYSIS"
    )

print("=" * 112)

if not all_metric_checks_pass:

    failed = [
        name
        for name, passed
        in metric_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage E1 metric checkpoint failed:\n- "
        + "\n- ".join(failed)
    )

## Stage F — Class-Level Performance and Error Structure

### Stage F1 — Compute Class Performance, Confusion Matrices, and All Nine Transitions

Preserve the full class-level predictive structure for each future window without interpreting the observed patterns.

For every F1–F6 window, this stage computes:

**Class-specific performance**
- support
- precision
- recall
- F1

**Confusion matrix**
- raw 3 × 3 counts using the frozen label order:
  - negative
  - neutral
  - positive

**Directed transitions**
- `negative → negative`
- `negative → neutral`
- `negative → positive`
- `neutral → negative`
- `neutral → neutral`
- `neutral → positive`
- `positive → negative`
- `positive → neutral`
- `positive → positive`

For each transition, save:

- raw count;
- true-class support;
- conditional transition rate.

The conditional rate is defined as:

\[
P(\hat{Y}=j\mid Y=i,F_k)
=
\frac{N(Y=i,\hat{Y}=j,F_k)}
     {N(Y=i,F_k)}
\]

This stage creates diagnostic evidence only.  
The temporal behavior of specific failure modes, including `negative → neutral`, is reserved for Day17.

Outputs:

- `future_class_performance.csv`
- `future_confusion_matrices.csv`
- `future_error_transitions.csv`

In [ ]:
# ============================================================
# DAY 16 — CELL 17
# Stage F1:
# Class performance + confusion matrices + 9 transitions
# ============================================================

from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
)

print("=" * 116)
print("DAY 16 — STAGE F1: CLASS PERFORMANCE AND CONFUSION STRUCTURE")
print("=" * 116)

# ------------------------------------------------------------
# 1. Authoritative evidence layer
# ------------------------------------------------------------

future_predictions = pd.read_csv(
    DAY16_DIR / "future_predictions.csv"
)

LABEL_ORDER = [
    "negative",
    "neutral",
    "positive",
]

# ------------------------------------------------------------
# 2. Input integrity
# ------------------------------------------------------------

stage_f_input_checks = {
    "prediction_n_20002":
        len(future_predictions) == 20_002,

    "six_windows_present":
        set(
            future_predictions["period"].unique()
        ) == set(EXPECTED_WINDOWS),

    "all_true_labels_valid":
        set(
            future_predictions["y_true"].unique()
        ) == set(LABEL_ORDER),

    "all_predictions_valid":
        set(
            future_predictions["y_pred"].unique()
        ).issubset(
            set(LABEL_ORDER)
        ),
}

for name, passed in stage_f_input_checks.items():
    if not passed:
        raise RuntimeError(
            f"Stage F input check failed: {name}"
        )

# ------------------------------------------------------------
# 3. Containers
# ------------------------------------------------------------

class_performance_records = []
confusion_records = []
transition_records = []

# ------------------------------------------------------------
# 4. Compute window-specific class structure
# ------------------------------------------------------------

for period in EXPECTED_WINDOWS:

    period_df = (
        future_predictions.loc[
            future_predictions["period"] == period
        ]
        .copy()
    )

    y_true = period_df["y_true"].astype(str)
    y_pred = period_df["y_pred"].astype(str)

    period_type = (
        period_df["period_type"]
        .dropna()
        .unique()
        .tolist()
    )

    if len(period_type) != 1:
        raise RuntimeError(
            f"Inconsistent period_type in {period}."
        )

    period_type = period_type[0]

    # --------------------------------------------------------
    # Class-specific precision / recall / F1 / support
    # --------------------------------------------------------

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=LABEL_ORDER,
            zero_division=0,
        )
    )

    for i, label in enumerate(LABEL_ORDER):

        class_performance_records.append(
            {
                "period": period,
                "period_type": period_type,
                "class": label,

                "support": int(
                    support[i]
                ),

                "precision": float(
                    precision[i]
                ),

                "recall": float(
                    recall[i]
                ),

                "f1": float(
                    f1[i]
                ),
            }
        )

    # --------------------------------------------------------
    # 3 × 3 confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
    )

    if int(cm.sum()) != len(period_df):
        raise RuntimeError(
            f"Confusion matrix total mismatch in {period}."
        )

    # --------------------------------------------------------
    # Save tidy confusion counts
    # --------------------------------------------------------

    for true_i, true_label in enumerate(
        LABEL_ORDER
    ):

        true_class_support = int(
            cm[true_i, :].sum()
        )

        for pred_i, pred_label in enumerate(
            LABEL_ORDER
        ):

            count = int(
                cm[true_i, pred_i]
            )

            conditional_rate = (
                count / true_class_support
                if true_class_support > 0
                else 0.0
            )

            confusion_records.append(
                {
                    "period": period,
                    "period_type": period_type,
                    "true_label": true_label,
                    "pred_label": pred_label,
                    "count": count,
                }
            )

            transition_records.append(
                {
                    "period": period,
                    "period_type": period_type,

                    "true_label":
                        true_label,

                    "pred_label":
                        pred_label,

                    "transition":
                        f"{true_label}->{pred_label}",

                    "count":
                        count,

                    "true_class_support":
                        true_class_support,

                    "conditional_rate":
                        float(
                            conditional_rate
                        ),
                }
            )

# ------------------------------------------------------------
# 5. Materialize tables
# ------------------------------------------------------------

future_class_performance = pd.DataFrame(
    class_performance_records
)

future_confusion_matrices = pd.DataFrame(
    confusion_records
)

future_error_transitions = pd.DataFrame(
    transition_records
)

# ------------------------------------------------------------
# 6. Structural integrity checks
# ------------------------------------------------------------

stage_f_checks = {}

# 6 periods × 3 classes
stage_f_checks[
    "class_performance_rows_18"
] = (
    len(future_class_performance)
    == 6 * 3
)

# 6 periods × 9 confusion cells
stage_f_checks[
    "confusion_rows_54"
] = (
    len(future_confusion_matrices)
    == 6 * 9
)

stage_f_checks[
    "transition_rows_54"
] = (
    len(future_error_transitions)
    == 6 * 9
)

stage_f_checks[
    "nine_transitions_per_window"
] = (
    future_error_transitions
    .groupby("period")
    .size()
    .eq(9)
    .all()
)

stage_f_checks[
    "class_support_sums_match_window_n"
] = all(
    int(
        future_class_performance.loc[
            future_class_performance[
                "period"
            ] == period,
            "support",
        ].sum()
    )
    ==
    int(
        (
            future_predictions[
                "period"
            ] == period
        ).sum()
    )
    for period in EXPECTED_WINDOWS
)

stage_f_checks[
    "confusion_totals_match_window_n"
] = all(
    int(
        future_confusion_matrices.loc[
            future_confusion_matrices[
                "period"
            ] == period,
            "count",
        ].sum()
    )
    ==
    int(
        (
            future_predictions[
                "period"
            ] == period
        ).sum()
    )
    for period in EXPECTED_WINDOWS
)

# For each true class within each window,
# the 3 conditional rates should sum to 1.
transition_rate_sums = (
    future_error_transitions
    .groupby(
        [
            "period",
            "true_label",
        ]
    )["conditional_rate"]
    .sum()
)

stage_f_checks[
    "conditional_rates_sum_to_one"
] = bool(
    np.allclose(
        transition_rate_sums.values,
        1.0,
        atol=1e-12,
    )
)

stage_f_checks[
    "all_metric_values_valid"
] = (
    future_class_performance[
        [
            "precision",
            "recall",
            "f1",
        ]
    ]
    .apply(
        lambda s: s.between(
            0,
            1
        ).all()
    )
    .all()
)

# ------------------------------------------------------------
# 7. Cross-check class recalls against Stage E
# ------------------------------------------------------------

recall_pivot = (
    future_class_performance
    .pivot(
        index="period",
        columns="class",
        values="recall",
    )
)

stage_e_lookup = (
    future_window_metrics
    .set_index("period")
)

recall_crosscheck = True

for period in EXPECTED_WINDOWS:

    recall_crosscheck &= np.isclose(
        recall_pivot.loc[
            period,
            "negative"
        ],
        stage_e_lookup.loc[
            period,
            "negative_recall"
        ],
    )

    recall_crosscheck &= np.isclose(
        recall_pivot.loc[
            period,
            "neutral"
        ],
        stage_e_lookup.loc[
            period,
            "neutral_recall"
        ],
    )

    recall_crosscheck &= np.isclose(
        recall_pivot.loc[
            period,
            "positive"
        ],
        stage_e_lookup.loc[
            period,
            "positive_recall"
        ],
    )

stage_f_checks[
    "recalls_match_stage_e"
] = bool(
    recall_crosscheck
)

# ------------------------------------------------------------
# 8. Cross-check priority failure mode directly from records
# ------------------------------------------------------------

neg_neu_direct = (
    future_predictions.loc[
        future_predictions[
            "y_true"
        ].eq("negative")
    ]
    .groupby("period")["y_pred"]
    .apply(
        lambda s:
            float(
                s.eq("neutral").mean()
            )
    )
)

neg_neu_transition = (
    future_error_transitions.loc[
        future_error_transitions[
            "transition"
        ].eq(
            "negative->neutral"
        )
    ]
    .set_index("period")[
        "conditional_rate"
    ]
)

stage_f_checks[
    "negative_to_neutral_crosscheck"
] = all(
    np.isclose(
        neg_neu_direct.loc[period],
        neg_neu_transition.loc[period],
    )
    for period in EXPECTED_WINDOWS
)

# ------------------------------------------------------------
# 9. Save formal artifacts
# ------------------------------------------------------------

class_performance_path = (
    DAY16_DIR
    / "future_class_performance.csv"
)

confusion_path = (
    DAY16_DIR
    / "future_confusion_matrices.csv"
)

transition_path = (
    DAY16_DIR
    / "future_error_transitions.csv"
)

future_class_performance.to_csv(
    class_performance_path,
    index=False,
)

future_confusion_matrices.to_csv(
    confusion_path,
    index=False,
)

future_error_transitions.to_csv(
    transition_path,
    index=False,
)

stage_f_checks[
    "class_performance_saved"
] = class_performance_path.exists()

stage_f_checks[
    "confusion_matrices_saved"
] = confusion_path.exists()

stage_f_checks[
    "error_transitions_saved"
] = transition_path.exists()

# ------------------------------------------------------------
# 10. Compact reporting tables
# ------------------------------------------------------------

print("\n[CLASS PERFORMANCE]")

display(
    future_class_performance
)

print("\n" + "-" * 116)
print("[CONFUSION MATRICES — RAW COUNTS]")

for period in EXPECTED_WINDOWS:

    period_cm = (
        future_confusion_matrices.loc[
            future_confusion_matrices[
                "period"
            ] == period
        ]
        .pivot(
            index="true_label",
            columns="pred_label",
            values="count",
        )
        .reindex(
            index=LABEL_ORDER,
            columns=LABEL_ORDER,
        )
    )

    print(f"\n{period}")
    display(period_cm)

# ------------------------------------------------------------
# 11. Descriptive transition table
#     No temporal diagnosis.
# ------------------------------------------------------------

print("\n" + "-" * 116)
print("[ERROR TRANSITIONS — CONDITIONAL RATES]")

transition_rate_table = (
    future_error_transitions
    .pivot(
        index="period",
        columns="transition",
        values="conditional_rate",
    )
    .reindex(
        EXPECTED_WINDOWS
    )
)

display(
    transition_rate_table
)

# ------------------------------------------------------------
# 12. Final checks
# ------------------------------------------------------------

print("\n" + "-" * 116)
print("[STAGE F1 CHECKS]")

for name, passed in stage_f_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

all_stage_f_checks_pass = all(
    stage_f_checks.values()
)

print("-" * 116)

print(
    f"Class performance   : "
    f"{class_performance_path}"
)

print(
    f"Confusion matrices  : "
    f"{confusion_path}"
)

print(
    f"Error transitions   : "
    f"{transition_path}"
)

if all_stage_f_checks_pass:

    print(
        "STAGE F1 STATUS : PASS — "
        "CLASS PERFORMANCE AND ERROR STRUCTURE FROZEN"
    )

else:

    print(
        "STAGE F1 STATUS : FAIL — "
        "STOP BEFORE COMPOSITION TABLES"
    )

print("=" * 116)

if not all_stage_f_checks_pass:

    failed = [
        name
        for name, passed
        in stage_f_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage F1 checkpoint failed:\n- "
        + "\n- ".join(failed)
    )

## Stage G — Prediction Distribution and Source Composition

### Stage G1 — Record Prediction Behavior and Source Composition

Preserve the descriptive composition information required for Day17 without interpreting its association with performance.

For each future window F1–F6, record:

**Prediction distribution**
- predicted negative count and share;
- predicted neutral count and share;
- predicted positive count and share.

**Observed benchmark-label distribution**
- true negative count and share;
- true neutral count and share;
- true positive count and share.

**Source composition**
- observed source count and share;
- Day14 frozen expected source count;
- exact count-match flag.

The source composition is recomputed from the exact Day16 future population and cross-checked against the Day14 frozen manifest.

No relationship between source mix, label mix, prediction mix, or predictive performance is analyzed in this stage.

Outputs:

- `future_prediction_distribution.csv`
- `future_source_composition.csv`

In [ ]:
# ============================================================
# DAY 16 — CELL 18
# Stage G1:
# Prediction distribution + source composition
# ============================================================

print("=" * 116)
print("DAY 16 — STAGE G1: PREDICTION DISTRIBUTION AND SOURCE COMPOSITION")
print("=" * 116)

# ------------------------------------------------------------
# 1. Reload authoritative prediction layer
# ------------------------------------------------------------

future_predictions = pd.read_csv(
    DAY16_DIR / "future_predictions.csv"
)

LABEL_ORDER = [
    "negative",
    "neutral",
    "positive",
]

# ------------------------------------------------------------
# 2. Prediction + true-label distribution
# ------------------------------------------------------------

distribution_records = []

for period in EXPECTED_WINDOWS:

    period_df = (
        future_predictions.loc[
            future_predictions["period"] == period
        ]
        .copy()
    )

    period_type_values = (
        period_df["period_type"]
        .dropna()
        .unique()
        .tolist()
    )

    if len(period_type_values) != 1:
        raise RuntimeError(
            f"Inconsistent period_type in {period}."
        )

    period_type = period_type_values[0]

    n_period = len(period_df)

    pred_counts = (
        period_df["y_pred"]
        .value_counts()
        .reindex(
            LABEL_ORDER,
            fill_value=0,
        )
    )

    true_counts = (
        period_df["y_true"]
        .value_counts()
        .reindex(
            LABEL_ORDER,
            fill_value=0,
        )
    )

    for label in LABEL_ORDER:

        distribution_records.append(
            {
                "period":
                    period,

                "period_type":
                    period_type,

                "label":
                    label,

                "n":
                    int(n_period),

                "predicted_count":
                    int(
                        pred_counts[label]
                    ),

                "predicted_share":
                    float(
                        pred_counts[label]
                        / n_period
                    ),

                "true_count":
                    int(
                        true_counts[label]
                    ),

                "true_share":
                    float(
                        true_counts[label]
                        / n_period
                    ),
            }
        )

future_prediction_distribution = pd.DataFrame(
    distribution_records
)

# ------------------------------------------------------------
# 3. Source composition from exact future population
# ------------------------------------------------------------

observed_source_composition = (
    future_predictions
    .groupby(
        [
            "period",
            "period_type",
            "source",
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="count"
    )
)

# Add zero-count source/window combinations so the
# Day16 table is directly comparable with Day14.
all_sources = sorted(
    source_composition_by_split[
        "channel_username"
    ].unique()
)

source_grid = pd.MultiIndex.from_product(
    [
        EXPECTED_WINDOWS,
        all_sources,
    ],
    names=[
        "period",
        "source",
    ],
).to_frame(index=False)

period_type_lookup = (
    future_predictions[
        [
            "period",
            "period_type",
        ]
    ]
    .drop_duplicates()
)

future_source_composition = (
    source_grid
    .merge(
        period_type_lookup,
        on="period",
        how="left",
        validate="many_to_one",
    )
    .merge(
        observed_source_composition,
        on=[
            "period",
            "period_type",
            "source",
        ],
        how="left",
        validate="one_to_one",
    )
)

future_source_composition["count"] = (
    future_source_composition[
        "count"
    ]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 4. Add per-window n and observed share
# ------------------------------------------------------------

window_n_lookup = (
    future_predictions
    .groupby(
        "period",
        observed=True,
    )
    .size()
    .to_dict()
)

future_source_composition[
    "window_n"
] = future_source_composition[
    "period"
].map(
    window_n_lookup
).astype(int)

future_source_composition[
    "share"
] = (
    future_source_composition[
        "count"
    ]
    /
    future_source_composition[
        "window_n"
    ]
)

# ------------------------------------------------------------
# 5. Day14 frozen source expectations
# ------------------------------------------------------------

expected_source_manifest = (
    source_composition_by_split.loc[
        source_composition_by_split[
            "split"
        ].isin(
            EXPECTED_WINDOWS
        ),
        [
            "split",
            "channel_username",
            "channel_count",
            "channel_proportion",
            "split_n",
        ]
    ]
    .rename(
        columns={
            "split":
                "period",

            "channel_username":
                "source",

            "channel_count":
                "day14_expected_count",

            "channel_proportion":
                "day14_expected_share",

            "split_n":
                "day14_expected_window_n",
        }
    )
)

future_source_composition = (
    future_source_composition
    .merge(
        expected_source_manifest,
        on=[
            "period",
            "source",
        ],
        how="left",
        validate="one_to_one",
    )
)

future_source_composition[
    "day14_expected_count"
] = (
    future_source_composition[
        "day14_expected_count"
    ]
    .fillna(0)
    .astype(int)
)

future_source_composition[
    "count_match_flag"
] = (
    future_source_composition[
        "count"
    ]
    ==
    future_source_composition[
        "day14_expected_count"
    ]
)

future_source_composition[
    "share_match_flag"
] = np.isclose(
    future_source_composition[
        "share"
    ],
    future_source_composition[
        "day14_expected_share"
    ],
    atol=1e-12,
)

# ------------------------------------------------------------
# 6. Structural checks
# ------------------------------------------------------------

stage_g_checks = {}

stage_g_checks[
    "prediction_distribution_rows_18"
] = (
    len(
        future_prediction_distribution
    )
    == 6 * 3
)

stage_g_checks[
    "predicted_counts_sum_to_window_n"
] = all(
    int(
        future_prediction_distribution.loc[
            future_prediction_distribution[
                "period"
            ] == period,
            "predicted_count",
        ].sum()
    )
    ==
    int(
        (
            future_predictions[
                "period"
            ] == period
        ).sum()
    )
    for period in EXPECTED_WINDOWS
)

stage_g_checks[
    "true_counts_sum_to_window_n"
] = all(
    int(
        future_prediction_distribution.loc[
            future_prediction_distribution[
                "period"
            ] == period,
            "true_count",
        ].sum()
    )
    ==
    int(
        (
            future_predictions[
                "period"
            ] == period
        ).sum()
    )
    for period in EXPECTED_WINDOWS
)

stage_g_checks[
    "predicted_shares_sum_to_one"
] = bool(
    np.allclose(
        future_prediction_distribution
        .groupby(
            "period"
        )[
            "predicted_share"
        ]
        .sum()
        .values,
        1.0,
        atol=1e-12,
    )
)

stage_g_checks[
    "true_shares_sum_to_one"
] = bool(
    np.allclose(
        future_prediction_distribution
        .groupby(
            "period"
        )[
            "true_share"
        ]
        .sum()
        .values,
        1.0,
        atol=1e-12,
    )
)

stage_g_checks[
    "source_rows_48"
] = (
    len(
        future_source_composition
    )
    == 6 * len(all_sources)
)

stage_g_checks[
    "source_counts_sum_to_window_n"
] = all(
    int(
        future_source_composition.loc[
            future_source_composition[
                "period"
            ] == period,
            "count",
        ].sum()
    )
    ==
    int(
        (
            future_predictions[
                "period"
            ] == period
        ).sum()
    )
    for period in EXPECTED_WINDOWS
)

stage_g_checks[
    "source_shares_sum_to_one"
] = bool(
    np.allclose(
        future_source_composition
        .groupby(
            "period"
        )[
            "share"
        ]
        .sum()
        .values,
        1.0,
        atol=1e-12,
    )
)

stage_g_checks[
    "all_source_counts_match_day14"
] = bool(
    future_source_composition[
        "count_match_flag"
    ].all()
)

stage_g_checks[
    "all_source_shares_match_day14"
] = bool(
    future_source_composition[
        "share_match_flag"
    ].all()
)

# ------------------------------------------------------------
# 7. Save artifacts
# ------------------------------------------------------------

prediction_distribution_path = (
    DAY16_DIR
    / "future_prediction_distribution.csv"
)

source_composition_path = (
    DAY16_DIR
    / "future_source_composition.csv"
)

future_prediction_distribution.to_csv(
    prediction_distribution_path,
    index=False,
)

future_source_composition.to_csv(
    source_composition_path,
    index=False,
)

stage_g_checks[
    "prediction_distribution_saved"
] = (
    prediction_distribution_path.exists()
)

stage_g_checks[
    "source_composition_saved"
] = (
    source_composition_path.exists()
)

# ------------------------------------------------------------
# 8. Reporting
# ------------------------------------------------------------

print("\n[PREDICTION / TRUE-LABEL DISTRIBUTION]")

prediction_distribution_wide = (
    future_prediction_distribution
    .pivot(
        index="period",
        columns="label",
        values=[
            "predicted_share",
            "true_share",
        ],
    )
)

display(
    prediction_distribution_wide
)

print("\n" + "-" * 116)
print("[SOURCE COMPOSITION]")

display(
    future_source_composition[
        [
            "period",
            "period_type",
            "source",
            "count",
            "share",
            "day14_expected_count",
            "day14_expected_share",
            "count_match_flag",
            "share_match_flag",
        ]
    ]
)

print("\n" + "-" * 116)
print("[STAGE G1 CHECKS]")

for name, passed in stage_g_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

all_stage_g_checks_pass = all(
    stage_g_checks.values()
)

print("-" * 116)

print(
    f"Prediction distribution : "
    f"{prediction_distribution_path}"
)

print(
    f"Source composition      : "
    f"{source_composition_path}"
)

if all_stage_g_checks_pass:

    print(
        "STAGE G1 STATUS : PASS — "
        "PREDICTION AND SOURCE COMPOSITION FROZEN"
    )

else:

    print(
        "STAGE G1 STATUS : FAIL — "
        "STOP BEFORE TEMPORAL CURVE TABLE"
    )

print("=" * 116)

if not all_stage_g_checks_pass:

    failed = [
        name
        for name, passed
        in stage_g_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage G1 checkpoint failed:\n- "
        + "\n- ".join(failed)
    )

## Stage H — Temporal Performance Curve

### Stage H1 — Build the Figure-Ready Temporal Performance Table

Combine the frozen historical-validation baseline with the strictly future F1–F6 performance measurements.

The temporal sequence is represented as:

\[
V \rightarrow F1 \rightarrow F2 \rightarrow F3 \rightarrow F4 \rightarrow F5 \rightarrow F6
\]

but these periods are **not treated as seven homogeneous time bins**.

Their roles are explicitly distinguished:

- **Historical Validation (V)**
  - pooled Nov–Dec 2025;
  - historical reference anchor only.

- **F1–F5**
  - full monthly future windows;
  - primary future temporal sequence.

- **F6**
  - partial June 2026 window;
  - retained as a partial-window sensitivity point.

The table contains the frozen Day16 plotting metrics:

- Macro-F1;
- Accuracy;
- Weighted-F1;
- Negative recall;
- Neutral recall;
- Positive recall.

No smoothing, temporal regression, trend fitting, or causal interpretation is performed.

Output:

`temporal_performance_curve.csv`

In [ ]:
# ============================================================
# DAY 16 — CELL 19
# Stage H1: build temporal performance curve table
# ============================================================

print("=" * 116)
print("DAY 16 — STAGE H1: TEMPORAL PERFORMANCE CURVE TABLE")
print("=" * 116)

# ------------------------------------------------------------
# 1. Reload frozen future metrics
# ------------------------------------------------------------

future_window_metrics = pd.read_csv(
    DAY16_DIR / "future_window_metrics.csv"
)

# ------------------------------------------------------------
# 2. Historical Validation anchor
# ------------------------------------------------------------

historical_validation_start = (
    historical_validation_manifest[
        "start_timestamp_utc"
    ]
)

historical_validation_end = (
    historical_validation_manifest[
        "end_timestamp_utc"
    ]
)

historical_anchor = {
    "period":
        "Historical Validation",

    "period_short":
        "V",

    "period_order":
        0,

    "period_type":
        "historical_pooled",

    "start_utc":
        historical_validation_start,

    "end_utc":
        historical_validation_end,

    "n":
        int(
            historical_validation_metrics[
                "n_observations"
            ]
        ),

    "macro_f1":
        float(
            historical_validation_metrics[
                "macro_f1"
            ]
        ),

    "accuracy":
        float(
            historical_validation_metrics[
                "accuracy"
            ]
        ),

    "weighted_f1":
        float(
            historical_validation_metrics[
                "weighted_f1"
            ]
        ),

    "negative_recall":
        float(
            historical_validation_metrics[
                "class_recall"
            ]["negative"]
        ),

    "neutral_recall":
        float(
            historical_validation_metrics[
                "class_recall"
            ]["neutral"]
        ),

    "positive_recall":
        float(
            historical_validation_metrics[
                "class_recall"
            ]["positive"]
        ),

    "interpretive_role":
        "pooled_historical_reference_anchor",
}

# ------------------------------------------------------------
# 3. Future rows
# ------------------------------------------------------------

future_curve = (
    future_window_metrics
    .copy()
)

future_curve["period_short"] = (
    future_curve["period"]
)

future_curve["period_order"] = (
    future_curve["period"]
    .str.replace("F", "", regex=False)
    .astype(int)
)

future_curve["interpretive_role"] = (
    future_curve["period_type"].map(
        {
            "future_month":
                "full_future_month",

            "future_partial":
                "partial_future_sensitivity_point",
        }
    )
)

# ------------------------------------------------------------
# 4. Combine V + F1–F6
# ------------------------------------------------------------

temporal_performance_curve = pd.concat(
    [
        pd.DataFrame(
            [historical_anchor]
        ),

        future_curve[
            [
                "period",
                "period_short",
                "period_order",
                "period_type",
                "start_utc",
                "end_utc",
                "n",
                "macro_f1",
                "accuracy",
                "weighted_f1",
                "negative_recall",
                "neutral_recall",
                "positive_recall",
                "interpretive_role",
            ]
        ],
    ],
    ignore_index=True,
)

temporal_performance_curve = (
    temporal_performance_curve
    .sort_values(
        "period_order"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Frozen comparison deltas
#    Descriptive only — no diagnosis.
# ------------------------------------------------------------

v_macro_f1 = float(
    temporal_performance_curve.loc[
        temporal_performance_curve[
            "period_short"
        ] == "V",
        "macro_f1",
    ].iloc[0]
)

f1_macro_f1 = float(
    temporal_performance_curve.loc[
        temporal_performance_curve[
            "period_short"
        ] == "F1",
        "macro_f1",
    ].iloc[0]
)

temporal_performance_curve[
    "macro_f1_change_vs_V"
] = (
    temporal_performance_curve[
        "macro_f1"
    ]
    - v_macro_f1
)

temporal_performance_curve[
    "macro_f1_change_vs_F1"
] = (
    temporal_performance_curve[
        "macro_f1"
    ]
    - f1_macro_f1
)

# V has no meaningful F1 comparison role
temporal_performance_curve.loc[
    temporal_performance_curve[
        "period_short"
    ] == "V",
    "macro_f1_change_vs_F1",
] = np.nan

# ------------------------------------------------------------
# 6. Integrity checks
# ------------------------------------------------------------

stage_h_checks = {}

stage_h_checks[
    "seven_rows_V_plus_F1_to_F6"
] = (
    len(
        temporal_performance_curve
    ) == 7
)

stage_h_checks[
    "period_order_exact"
] = (
    temporal_performance_curve[
        "period_short"
    ].tolist()
    ==
    [
        "V",
        "F1",
        "F2",
        "F3",
        "F4",
        "F5",
        "F6",
    ]
)

stage_h_checks[
    "historical_validation_n_5859"
] = (
    int(
        temporal_performance_curve.loc[
            temporal_performance_curve[
                "period_short"
            ] == "V",
            "n",
        ].iloc[0]
    )
    == 5_859
)

stage_h_checks[
    "historical_anchor_typed_correctly"
] = (
    temporal_performance_curve.loc[
        temporal_performance_curve[
            "period_short"
        ] == "V",
        "period_type",
    ].iloc[0]
    == "historical_pooled"
)

stage_h_checks[
    "F1_to_F5_are_full_months"
] = (
    temporal_performance_curve.loc[
        temporal_performance_curve[
            "period_short"
        ].isin(
            [
                "F1",
                "F2",
                "F3",
                "F4",
                "F5",
            ]
        ),
        "period_type",
    ]
    .eq(
        "future_month"
    )
    .all()
)

stage_h_checks[
    "F6_marked_partial"
] = (
    temporal_performance_curve.loc[
        temporal_performance_curve[
            "period_short"
        ] == "F6",
        "period_type",
    ].iloc[0]
    == "future_partial"
)

stage_h_checks[
    "future_n_total_20002"
] = (
    int(
        temporal_performance_curve.loc[
            temporal_performance_curve[
                "period_short"
            ].isin(
                EXPECTED_WINDOWS
            ),
            "n",
        ].sum()
    )
    == 20_002
)

stage_h_checks[
    "V_metrics_match_day15"
] = (
    np.isclose(
        temporal_performance_curve.loc[
            temporal_performance_curve[
                "period_short"
            ] == "V",
            "macro_f1",
        ].iloc[0],
        historical_validation_metrics[
            "macro_f1"
        ],
    )
    and
    np.isclose(
        temporal_performance_curve.loc[
            temporal_performance_curve[
                "period_short"
            ] == "V",
            "accuracy",
        ].iloc[0],
        historical_validation_metrics[
            "accuracy"
        ],
    )
)

stage_h_checks[
    "future_metrics_match_stage_e"
] = all(
    np.isclose(
        temporal_performance_curve.loc[
            temporal_performance_curve[
                "period_short"
            ] == period,
            "macro_f1",
        ].iloc[0],

        future_window_metrics.loc[
            future_window_metrics[
                "period"
            ] == period,
            "macro_f1",
        ].iloc[0],
    )
    for period in EXPECTED_WINDOWS
)

stage_h_checks[
    "all_metrics_in_valid_range"
] = (
    temporal_performance_curve[
        [
            "macro_f1",
            "accuracy",
            "weighted_f1",
            "negative_recall",
            "neutral_recall",
            "positive_recall",
        ]
    ]
    .apply(
        lambda x:
            x.between(
                0,
                1
            ).all()
    )
    .all()
)

# ------------------------------------------------------------
# 7. Save formal artifact
# ------------------------------------------------------------

temporal_curve_path = (
    DAY16_DIR
    / "temporal_performance_curve.csv"
)

temporal_performance_curve.to_csv(
    temporal_curve_path,
    index=False,
)

stage_h_checks[
    "temporal_curve_saved"
] = (
    temporal_curve_path.exists()
)

# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("\n[TEMPORAL PERFORMANCE CURVE TABLE]")

display(
    temporal_performance_curve[
        [
            "period_short",
            "period_type",
            "n",
            "macro_f1",
            "accuracy",
            "weighted_f1",
            "negative_recall",
            "neutral_recall",
            "positive_recall",
            "macro_f1_change_vs_V",
            "macro_f1_change_vs_F1",
            "interpretive_role",
        ]
    ]
)

print("\n" + "-" * 116)
print("[STAGE H1 CHECKS]")

for name, passed in stage_h_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

all_stage_h_checks_pass = all(
    stage_h_checks.values()
)

print("-" * 116)

print(
    f"Temporal curve artifact : "
    f"{temporal_curve_path}"
)

if all_stage_h_checks_pass:

    print(
        "STAGE H1 STATUS : PASS — "
        "TEMPORAL PERFORMANCE TABLE FROZEN"
    )

else:

    print(
        "STAGE H1 STATUS : FAIL — "
        "STOP BEFORE FIGURE GENERATION"
    )

print("=" * 116)

if not all_stage_h_checks_pass:

    failed = [
        name
        for name, passed
        in stage_h_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage H1 checkpoint failed:\n- "
        + "\n- ".join(failed)
    )

### Stage H2 — Generate Frozen Temporal Performance Figures

Visualize the frozen Day16 temporal performance measurements without smoothing, trend fitting, or causal interpretation.

Three figures are generated:

1. **Primary endpoint**
   - Macro-F1

2. **Supportive endpoint**
   - Accuracy

3. **Supportive class-specific endpoint**
   - Negative recall

Figure semantics:

- `V` = pooled Nov–Dec historical-validation reference anchor;
- `F1–F5` = full future monthly windows;
- `F6` = partial June sensitivity point.

The figures preserve the observed non-monotonic structure exactly as measured.

No regression line, moving average, interpolation, or temporal smoothing is added.

In [ ]:
# ============================================================
# DAY 16 — CELL 20
# Stage H2: generate frozen temporal performance figures
# ============================================================

import matplotlib.pyplot as plt

print("=" * 112)
print("DAY 16 — STAGE H2: TEMPORAL PERFORMANCE FIGURES")
print("=" * 112)

# ------------------------------------------------------------
# 1. Reload frozen curve table
# ------------------------------------------------------------

curve_df = pd.read_csv(
    DAY16_DIR / "temporal_performance_curve.csv"
)

period_order = [
    "V",
    "F1",
    "F2",
    "F3",
    "F4",
    "F5",
    "F6",
]

curve_df = (
    curve_df
    .set_index("period_short")
    .loc[period_order]
    .reset_index()
)

x_labels = curve_df["period_short"].tolist()
x_positions = list(range(len(x_labels)))

# ------------------------------------------------------------
# 2. Helper for frozen figure generation
# ------------------------------------------------------------

def plot_day16_curve(
    metric_column,
    y_label,
    title,
    output_filename,
):
    plt.figure(figsize=(9, 5.5))

    plt.plot(
        x_positions,
        curve_df[metric_column],
        marker="o",
        linewidth=2,
    )

    plt.xticks(
        x_positions,
        x_labels,
    )

    plt.xlabel("Evaluation period")
    plt.ylabel(y_label)
    plt.title(title)

    plt.grid(
        axis="y",
        alpha=0.25,
    )

    # ----------------------------------------
    # Annotate historical pooled anchor
    # ----------------------------------------

    v_row = curve_df.loc[
        curve_df["period_short"] == "V"
    ].iloc[0]

    plt.annotate(
        "Pooled Nov–Dec baseline",
        xy=(
            0,
            v_row[metric_column],
        ),
        xytext=(
            0.4,
            v_row[metric_column] + 0.015,
        ),
        arrowprops=dict(
            arrowstyle="->",
        ),
        fontsize=9,
    )

    # ----------------------------------------
    # Annotate partial F6
    # ----------------------------------------

    f6_idx = x_labels.index("F6")

    f6_row = curve_df.loc[
        curve_df["period_short"] == "F6"
    ].iloc[0]

    plt.annotate(
        "Partial June",
        xy=(
            f6_idx,
            f6_row[metric_column],
        ),
        xytext=(
            f6_idx - 1.2,
            f6_row[metric_column] + 0.015,
        ),
        arrowprops=dict(
            arrowstyle="->",
        ),
        fontsize=9,
    )

    plt.tight_layout()

    output_path = (
        DAY16_DIR
        / output_filename
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    return output_path


# ------------------------------------------------------------
# 3. Primary endpoint — Macro-F1
# ------------------------------------------------------------

macro_f1_figure_path = plot_day16_curve(
    metric_column="macro_f1",
    y_label="Macro-F1",
    title=(
        "Day16 Strict Future Temporal Generalization — Macro-F1"
    ),
    output_filename="temporal_macro_f1_curve.png",
)

# ------------------------------------------------------------
# 4. Supportive endpoint — Accuracy
# ------------------------------------------------------------

accuracy_figure_path = plot_day16_curve(
    metric_column="accuracy",
    y_label="Accuracy",
    title=(
        "Day16 Strict Future Temporal Generalization — Accuracy"
    ),
    output_filename="temporal_accuracy_curve.png",
)

# ------------------------------------------------------------
# 5. Supportive endpoint — Negative Recall
# ------------------------------------------------------------

negative_recall_figure_path = plot_day16_curve(
    metric_column="negative_recall",
    y_label="Negative Recall",
    title=(
        "Day16 Strict Future Temporal Generalization — Negative Recall"
    ),
    output_filename="temporal_negative_recall_curve.png",
)

# ------------------------------------------------------------
# 6. Figure checks
# ------------------------------------------------------------

figure_checks = {
    "macro_f1_figure_saved":
        macro_f1_figure_path.exists(),

    "accuracy_figure_saved":
        accuracy_figure_path.exists(),

    "negative_recall_figure_saved":
        negative_recall_figure_path.exists(),
}

print("\n[FIGURE CHECKS]")

for name, passed in figure_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

all_figure_checks_pass = all(
    figure_checks.values()
)

print("-" * 112)

print(
    f"Macro-F1 figure       : "
    f"{macro_f1_figure_path}"
)

print(
    f"Accuracy figure       : "
    f"{accuracy_figure_path}"
)

print(
    f"Negative recall figure: "
    f"{negative_recall_figure_path}"
)

if all_figure_checks_pass:

    print(
        "STAGE H2 STATUS : PASS — "
        "TEMPORAL PERFORMANCE FIGURES SAVED"
    )

else:

    print(
        "STAGE H2 STATUS : FAIL — "
        "FIGURE GENERATION INCOMPLETE"
    )

print("=" * 112)

if not all_figure_checks_pass:

    failed = [
        name
        for name, passed
        in figure_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage H2 figure checkpoint failed:\n- "
        + "\n- ".join(failed)
    )

## Stage I — Freeze Day16 Outputs

### Stage I1 — Create the Final Day16 Summary

Freeze the completed Day16 measurement state before any Day17 diagnostic analysis begins.

The summary records:

- frozen model identity;
- chronological protocol status;
- exact future population;
- F1–F6 performance metrics;
- Historical Validation reference anchor;
- endpoint hierarchy;
- label-provenance caveat;
- no-adaptation confirmation;
- F6 partial-window status;
- completion status of the Day16 evidence package.

Day16 conclusions remain descriptive only.

No explanation is assigned to:

- temporal distance;
- source-composition changes;
- benchmark-label composition;
- prediction-distribution changes;
- specific confusion transitions.

Those questions are deferred to Day17.

Output:

`day16_summary.json`

In [ ]:
# ============================================================
# DAY 16 — CELL 21
# Stage I1: write final Day16 summary
# ============================================================

from datetime import datetime, timezone

print("=" * 116)
print("DAY 16 — STAGE I1: FREEZE FINAL DAY16 SUMMARY")
print("=" * 116)

# ------------------------------------------------------------
# 1. Reload authoritative Day16 artifacts
# ------------------------------------------------------------

future_window_metrics = pd.read_csv(
    DAY16_DIR / "future_window_metrics.csv"
)

future_class_performance = pd.read_csv(
    DAY16_DIR / "future_class_performance.csv"
)

future_error_transitions = pd.read_csv(
    DAY16_DIR / "future_error_transitions.csv"
)

future_prediction_distribution = pd.read_csv(
    DAY16_DIR / "future_prediction_distribution.csv"
)

future_source_composition = pd.read_csv(
    DAY16_DIR / "future_source_composition.csv"
)

temporal_performance_curve = pd.read_csv(
    DAY16_DIR / "temporal_performance_curve.csv"
)

# ------------------------------------------------------------
# 2. Build per-window metric dictionary
# ------------------------------------------------------------

future_metric_summary = {}

for _, row in future_window_metrics.iterrows():

    period = str(row["period"])

    future_metric_summary[period] = {
        "period_type":
            str(row["period_type"]),

        "n":
            int(row["n"]),

        "macro_f1":
            float(row["macro_f1"]),

        "accuracy":
            float(row["accuracy"]),

        "weighted_f1":
            float(row["weighted_f1"]),

        "negative_recall":
            float(row["negative_recall"]),

        "neutral_recall":
            float(row["neutral_recall"]),

        "positive_recall":
            float(row["positive_recall"]),
    }

# ------------------------------------------------------------
# 3. Historical validation anchor
# ------------------------------------------------------------

historical_validation_anchor = {
    "role":
        "pooled_historical_reference_anchor",

    "period":
        "2025-11-01 through 2025-12-31",

    "n":
        int(
            historical_validation_metrics[
                "n_observations"
            ]
        ),

    "macro_f1":
        float(
            historical_validation_metrics[
                "macro_f1"
            ]
        ),

    "accuracy":
        float(
            historical_validation_metrics[
                "accuracy"
            ]
        ),

    "weighted_f1":
        float(
            historical_validation_metrics[
                "weighted_f1"
            ]
        ),

    "negative_recall":
        float(
            historical_validation_metrics[
                "class_recall"
            ]["negative"]
        ),

    "neutral_recall":
        float(
            historical_validation_metrics[
                "class_recall"
            ]["neutral"]
        ),

    "positive_recall":
        float(
            historical_validation_metrics[
                "class_recall"
            ]["positive"]
        ),
}

# ------------------------------------------------------------
# 4. Day16 descriptive extrema
#    Allowed measurement-level statements only
# ------------------------------------------------------------

lowest_macro_row = (
    future_window_metrics.loc[
        future_window_metrics[
            "macro_f1"
        ].idxmin()
    ]
)

highest_macro_row = (
    future_window_metrics.loc[
        future_window_metrics[
            "macro_f1"
        ].idxmax()
    ]
)

descriptive_measurement_summary = {
    "future_macro_f1_non_monotonic":
        True,

    "lowest_observed_future_macro_f1_window":
        str(
            lowest_macro_row["period"]
        ),

    "lowest_observed_future_macro_f1":
        float(
            lowest_macro_row["macro_f1"]
        ),

    "highest_observed_future_macro_f1_window":
        str(
            highest_macro_row["period"]
        ),

    "highest_observed_future_macro_f1":
        float(
            highest_macro_row["macro_f1"]
        ),

    "interpretation_rule":
        (
            "These are descriptive observed extrema only. "
            "No causal or diagnostic explanation is assigned in Day16."
        ),
}

# ------------------------------------------------------------
# 5. Required formal artifacts
# ------------------------------------------------------------

required_day16_artifacts = {
    "future_inference_manifest.json":
        DAY16_DIR / "future_inference_manifest.json",

    "future_population_audit.json":
        DAY16_DIR / "future_population_audit.json",

    "future_predictions.csv":
        DAY16_DIR / "future_predictions.csv",

    "future_window_metrics.csv":
        DAY16_DIR / "future_window_metrics.csv",

    "future_class_performance.csv":
        DAY16_DIR / "future_class_performance.csv",

    "future_confusion_matrices.csv":
        DAY16_DIR / "future_confusion_matrices.csv",

    "future_error_transitions.csv":
        DAY16_DIR / "future_error_transitions.csv",

    "future_prediction_distribution.csv":
        DAY16_DIR / "future_prediction_distribution.csv",

    "future_source_composition.csv":
        DAY16_DIR / "future_source_composition.csv",

    "temporal_performance_curve.csv":
        DAY16_DIR / "temporal_performance_curve.csv",
}

figure_artifacts = {
    "temporal_macro_f1_curve.png":
        DAY16_DIR / "temporal_macro_f1_curve.png",

    "temporal_accuracy_curve.png":
        DAY16_DIR / "temporal_accuracy_curve.png",

    "temporal_negative_recall_curve.png":
        DAY16_DIR / "temporal_negative_recall_curve.png",
}

formal_artifact_status = {
    name: path.exists()
    for name, path in required_day16_artifacts.items()
}

figure_artifact_status = {
    name: path.exists()
    for name, path in figure_artifacts.items()
}

# ------------------------------------------------------------
# 6. Final scientific completion flags
# ------------------------------------------------------------

completion_flags = {
    "stage_a_frozen_protocol_verified":
        bool(all_checks_pass),

    "stage_b_benchmark_population_verified":
        bool(all_benchmark_checks_pass),

    "stage_b_future_population_verified":
        bool(all_future_checks_pass),

    "stage_c_runtime_preflight_pass":
        bool(all_preflight_checks_pass),

    "stage_c_frozen_model_loaded":
        bool(all_load_checks_pass),

    "stage_c_day15_inference_reproduced":
        bool(all_stage_c3_checks_pass),

    "stage_d_future_inference_complete":
        bool(all_prediction_checks_pass),

    "stage_e_window_metrics_complete":
        bool(all_metric_checks_pass),

    "stage_f_error_structure_complete":
        bool(all_stage_f_checks_pass),

    "stage_g_composition_tables_complete":
        bool(all_stage_g_checks_pass),

    "stage_h_temporal_curve_complete":
        bool(all_stage_h_checks_pass),

    "stage_h_figures_complete":
        bool(all_figure_checks_pass),
}

all_scientific_stages_pass = all(
    completion_flags.values()
)

all_formal_artifacts_present = all(
    formal_artifact_status.values()
)

all_figures_present = all(
    figure_artifact_status.values()
)

# ------------------------------------------------------------
# 7. Build final Day16 summary
# ------------------------------------------------------------

day16_summary = {
    "day":
        16,

    "project":
        "Financial Language Models under Temporal Distribution Shift",

    "status":
        (
            "PASS — STRICT FUTURE TEMPORAL GENERALIZATION MEASUREMENT COMPLETE"
            if (
                all_scientific_stages_pass
                and all_formal_artifacts_present
                and all_figures_present
            )
            else
            "INCOMPLETE"
        ),

    "scientific_role":
        "strict_future_temporal_generalization_measurement",

    "research_question":
        (
            "How does predictive performance change across "
            "strictly future temporal windows?"
        ),

    "model": {
        "model_id":
            "M_H",

        "base_model":
            frozen_model_manifest[
                "model_identity"
            ]["base_model"],

        "selected_checkpoint_step":
            int(
                frozen_model_manifest[
                    "checkpoint_selection"
                ]["selected_checkpoint_step"]
            ),

        "freeze_status":
            frozen_model_manifest[
                "model_identity"
            ]["freeze_status"],

        "adaptation_performed":
            False,

        "trainable_parameters_during_future_evaluation":
            0,
    },

    "chronological_protocol": {
        "historical_cutoff_utc":
            chronological_split_spec[
                "historical_cutoff"
            ]["cutoff_timestamp_utc"],

        "historical_train_n":
            13_539,

        "historical_validation_n":
            5_859,

        "strictly_future_n":
            20_002,

        "future_windows":
            [
                "F1",
                "F2",
                "F3",
                "F4",
                "F5",
                "F6",
            ],

        "F1_to_F5_role":
            "full_future_months",

        "F6_role":
            "partial_month_sensitivity_point",
    },

    "evaluation_protocol": {
        "primary_endpoint":
            "macro_f1",

        "secondary_endpoints":
            [
                "accuracy",
                "weighted_f1",
            ],

        "class_specific_metrics":
            [
                "precision",
                "recall",
                "f1",
                "support",
            ],

        "priority_failure_mode_retained_for_day17":
            "negative->neutral",

        "label_order":
            [
                "negative",
                "neutral",
                "positive",
            ],
    },

    "historical_validation_anchor":
        historical_validation_anchor,

    "future_window_metrics":
        future_metric_summary,

    "descriptive_measurement_summary":
        descriptive_measurement_summary,

    "future_prediction_integrity": {
        "n_predictions":
            20_002,

        "invalid_predictions":
            0,

        "observation_order_preserved":
            True,

        "all_future_windows_present":
            True,
    },

    "source_protocol_crosscheck": {
        "all_source_counts_match_day14":
            True,

        "all_source_shares_match_day14":
            True,
    },

    "label_provenance": {
        "label_type":
            "LLM-generated Dataset-B benchmark labels",

        "human_gold_labels":
            False,

        "interpretation_caveat":
            (
                "Performance metrics describe agreement with the frozen "
                "Dataset-B benchmark sentiment labels and should not be "
                "overstated as accuracy against conventional human gold labels."
            ),
    },

    "scientific_boundary": {
        "day16":
            "measurement",

        "day17":
            "diagnosis",

        "day18":
            "controlled_temporal_composition_robustness_intervention",

        "causal_claim_about_time":
            False,

        "interpretation_deferred_to_day17":
            True,
    },

    "completion_flags":
        completion_flags,

    "formal_artifact_status":
        formal_artifact_status,

    "figure_artifact_status":
        figure_artifact_status,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

# ------------------------------------------------------------
# 8. Save summary
# ------------------------------------------------------------

day16_summary_path = (
    DAY16_DIR
    / "day16_summary.json"
)

with open(
    day16_summary_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        day16_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )

# ------------------------------------------------------------
# 9. Summary checks
# ------------------------------------------------------------

summary_checks = {
    "all_scientific_stages_pass":
        all_scientific_stages_pass,

    "all_pre_summary_artifacts_present":
        all_formal_artifacts_present,

    "all_three_figures_present":
        all_figures_present,

    "summary_saved":
        day16_summary_path.exists(),

    "future_n_20002":
        day16_summary[
            "chronological_protocol"
        ]["strictly_future_n"] == 20_002,

    "adaptation_performed_false":
        day16_summary[
            "model"
        ]["adaptation_performed"] is False,

    "interpretation_deferred_to_day17":
        day16_summary[
            "scientific_boundary"
        ]["interpretation_deferred_to_day17"] is True,

    "label_caveat_present":
        bool(
            day16_summary[
                "label_provenance"
            ]["interpretation_caveat"]
        ),
}

all_summary_checks_pass = all(
    summary_checks.values()
)

# ------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------

print("\n[DAY16 DESCRIPTIVE MEASUREMENT SUMMARY]")

print(
    "Historical Validation Macro-F1 : "
    f"{historical_validation_anchor['macro_f1']:.6f}"
)

for period in EXPECTED_WINDOWS:

    print(
        f"{period} Macro-F1"
        f"{' ' * (3 - len(period))}: "
        f"{future_metric_summary[period]['macro_f1']:.6f}"
    )

print()

print(
    "Lowest observed future Macro-F1  : "
    f"{descriptive_measurement_summary['lowest_observed_future_macro_f1_window']} "
    f"({descriptive_measurement_summary['lowest_observed_future_macro_f1']:.6f})"
)

print(
    "Highest observed future Macro-F1 : "
    f"{descriptive_measurement_summary['highest_observed_future_macro_f1_window']} "
    f"({descriptive_measurement_summary['highest_observed_future_macro_f1']:.6f})"
)

print(
    "Temporal pattern                 : "
    "NON-MONOTONIC"
)

print(
    "Interpretation                   : "
    "DEFERRED TO DAY17"
)

print("\n" + "-" * 116)
print("[STAGE I1 CHECKS]")

for name, passed in summary_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

print("-" * 116)

print(
    f"Summary artifact : "
    f"{day16_summary_path}"
)

if all_summary_checks_pass:

    print(
        "STAGE I1 STATUS : PASS — "
        "DAY16 SUMMARY FROZEN"
    )

else:

    print(
        "STAGE I1 STATUS : FAIL — "
        "DO NOT CLOSE DAY16"
    )

print("=" * 116)

if not all_summary_checks_pass:

    failed = [
        name
        for name, passed
        in summary_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 Stage I1 summary checkpoint failed:\n- "
        + "\n- ".join(failed)
    )

### Stage I2 — Final Day16 Artifact Checkpoint

Verify that every protocol-required Day16 output exists and that the final measurement package is complete.

Required formal artifacts:

1. `future_inference_manifest.json`
2. `future_population_audit.json`
3. `future_predictions.csv`
4. `future_window_metrics.csv`
5. `future_class_performance.csv`
6. `future_confusion_matrices.csv`
7. `future_error_transitions.csv`
8. `future_prediction_distribution.csv`
9. `future_source_composition.csv`
10. `temporal_performance_curve.csv`
11. `day16_summary.json`

Required figures:

- `temporal_macro_f1_curve.png`
- `temporal_accuracy_curve.png`
- `temporal_negative_recall_curve.png`

Day16 closes only if all frozen protocol checks, inference checks, metric checks, composition checks, curve checks, summary checks, and artifact checks pass.

In [ ]:
# ============================================================
# DAY 16 — CELL 22
# Final artifact checkpoint and terminal status
# ============================================================

print("=" * 116)
print("DAY 16 — FINAL ARTIFACT CHECKPOINT")
print("=" * 116)

# ------------------------------------------------------------
# 1. Required formal artifacts
# ------------------------------------------------------------

required_artifacts = {
    "future_inference_manifest.json":
        DAY16_DIR / "future_inference_manifest.json",

    "future_population_audit.json":
        DAY16_DIR / "future_population_audit.json",

    "future_predictions.csv":
        DAY16_DIR / "future_predictions.csv",

    "future_window_metrics.csv":
        DAY16_DIR / "future_window_metrics.csv",

    "future_class_performance.csv":
        DAY16_DIR / "future_class_performance.csv",

    "future_confusion_matrices.csv":
        DAY16_DIR / "future_confusion_matrices.csv",

    "future_error_transitions.csv":
        DAY16_DIR / "future_error_transitions.csv",

    "future_prediction_distribution.csv":
        DAY16_DIR / "future_prediction_distribution.csv",

    "future_source_composition.csv":
        DAY16_DIR / "future_source_composition.csv",

    "temporal_performance_curve.csv":
        DAY16_DIR / "temporal_performance_curve.csv",

    "day16_summary.json":
        DAY16_DIR / "day16_summary.json",
}

# ------------------------------------------------------------
# 2. Required figure artifacts
# ------------------------------------------------------------

required_figures = {
    "temporal_macro_f1_curve.png":
        DAY16_DIR / "temporal_macro_f1_curve.png",

    "temporal_accuracy_curve.png":
        DAY16_DIR / "temporal_accuracy_curve.png",

    "temporal_negative_recall_curve.png":
        DAY16_DIR / "temporal_negative_recall_curve.png",
}

artifact_checks = {
    name: path.exists()
    for name, path in required_artifacts.items()
}

figure_checks_final = {
    name: path.exists()
    for name, path in required_figures.items()
}

# ------------------------------------------------------------
# 3. Final scientific acceptance criteria
# ------------------------------------------------------------

acceptance_checks = {
    "frozen_protocol_verified":
        bool(all_checks_pass),

    "benchmark_population_39400_verified":
        bool(all_benchmark_checks_pass),

    "future_population_20002_verified":
        bool(all_future_checks_pass),

    "frozen_runtime_preflight_pass":
        bool(all_preflight_checks_pass),

    "frozen_model_loaded":
        bool(all_load_checks_pass),

    "day15_inference_reproduced":
        bool(all_stage_c3_checks_pass),

    "strict_future_inference_complete":
        bool(all_prediction_checks_pass),

    "window_metrics_complete":
        bool(all_metric_checks_pass),

    "class_error_structure_complete":
        bool(all_stage_f_checks_pass),

    "prediction_source_composition_complete":
        bool(all_stage_g_checks_pass),

    "temporal_curve_complete":
        bool(all_stage_h_checks_pass),

    "figures_complete":
        bool(all_figure_checks_pass),

    "day16_summary_complete":
        bool(all_summary_checks_pass),

    "future_n_20002":
        len(
            pd.read_csv(
                DAY16_DIR / "future_predictions.csv"
            )
        ) == 20_002,

    "no_invalid_predictions":
        int(
            (
                ~pd.read_csv(
                    DAY16_DIR / "future_predictions.csv"
                )["prediction_valid"]
            ).sum()
        ) == 0,

    "no_adaptation_performed":
        day16_summary[
            "model"
        ]["adaptation_performed"] is False,

    "interpretation_deferred_to_day17":
        day16_summary[
            "scientific_boundary"
        ]["interpretation_deferred_to_day17"] is True,

    "label_provenance_caveat_present":
        bool(
            day16_summary[
                "label_provenance"
            ]["interpretation_caveat"]
        ),
}

# ------------------------------------------------------------
# 4. Report formal artifacts
# ------------------------------------------------------------

print("\n[REQUIRED FORMAL ARTIFACTS]")

for name, passed in artifact_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

print("\n" + "-" * 116)
print("[REQUIRED FIGURES]")

for name, passed in figure_checks_final.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

print("\n" + "-" * 116)
print("[FINAL ACCEPTANCE CRITERIA]")

for name, passed in acceptance_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<5} — {name}"
    )

# ------------------------------------------------------------
# 5. Terminal status
# ------------------------------------------------------------

all_artifacts_pass = all(
    artifact_checks.values()
)

all_figures_pass = all(
    figure_checks_final.values()
)

all_acceptance_pass = all(
    acceptance_checks.values()
)

day16_final_pass = (
    all_artifacts_pass
    and all_figures_pass
    and all_acceptance_pass
)

print("\n" + "-" * 116)

print(
    f"Formal artifacts present : "
    f"{sum(artifact_checks.values())}/{len(artifact_checks)}"
)

print(
    f"Figures present          : "
    f"{sum(figure_checks_final.values())}/{len(figure_checks_final)}"
)

print(
    f"Acceptance checks passed : "
    f"{sum(acceptance_checks.values())}/{len(acceptance_checks)}"
)

print("-" * 116)

if day16_final_pass:

    print(
        "DAY 16 STATUS: PASS — "
        "STRICT FUTURE TEMPORAL GENERALIZATION MEASUREMENT COMPLETE"
    )

    print()
    print(
        "Scientific boundary:"
    )
    print(
        "Day16 = measurement complete"
    )
    print(
        "Day17 = diagnosis not yet started"
    )

else:

    print(
        "DAY 16 STATUS: FAIL / INCOMPLETE"
    )

print("=" * 116)

if not day16_final_pass:

    failed_items = []

    failed_items += [
        f"artifact::{name}"
        for name, passed
        in artifact_checks.items()
        if not passed
    ]

    failed_items += [
        f"figure::{name}"
        for name, passed
        in figure_checks_final.items()
        if not passed
    ]

    failed_items += [
        f"acceptance::{name}"
        for name, passed
        in acceptance_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Day16 final checkpoint failed:\n- "
        + "\n- ".join(failed_items)
    )